In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import logging
import os
import re
import sys
import zipfile
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from IPython.display import display  # noqa: E402
from mne.stats import permutation_cluster_1samp_test  # noqa: E402
from scipy.stats import pearsonr, t as student_t, wilcoxon  # noqa: E402

from scripts.analysis_common import participant_label  # noqa: E402
from scripts.notebook_helpers import (  # noqa: E402
    WAVELET_FREQ_MAX,
    WAVELET_FREQ_MIN,
    WAVELET_N_FREQS,
    resolve_notebook_wavelet_cache_dir,
)
from src.analysis import assr_trials as at  # noqa: E402
from src.analysis import iva_quality  # noqa: E402
from src.analysis.isc import FREQUENCY_BANDS  # noqa: E402
from src.analysis.iva_condition_comparison import slice_to_band  # noqa: E402
from src.analysis.wavelet_ica import zscore_by_time  # noqa: E402
from src.definitions.constants import AssrEpoch, ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    REAL_CONDITIONS,
    ConditionVariants,
    CoordinateSystems,
    ExclusionCategories,
    ExperimentNames,
    IvaVariants,
    MusicTypeVariants,
    PreprocessedDataVariants,
    SingleDataMetadata,
    SpectrumTypeVariants,
)
from src.filtering.dataset_filter import DatasetFilter  # noqa: E402
from src.io.iva_store import list_iva_results, load_iva_components  # noqa: E402
from src.io.loading import assr_electrode_mask  # noqa: E402
from src.preprocessing.pipeline import DatasetHandler  # noqa: E402
from src.visualization.iva_condition_plots import (  # noqa: E402
    plot_condition_mean_tf_maps,
    plot_condition_mean_topomaps,
    plot_participant_condition_tf_maps,
    plot_participant_condition_topomaps,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
%matplotlib inline
print("Setup complete.")

# Analysis of the Stored IVA Components — Both Conditions on the Subject Axis

Reads the components that
[`scripts/run_iva_condition_comparison.py`](../../scripts/run_iva_condition_comparison.py)
wrote with `--store_components` and analyses them. **Nothing is decomposed here.**
The decomposition is the expensive part — a per-recording PCA plus `iva_g` over a
tensor of tens of gigabytes, which belongs on a node
([`run_condition_comparison.pbs`](../../jobs/metacentrum/06-iva-condition-comparison/run_condition_comparison.pbs))
— and its figures are lossy summaries of it: a condition-mean grid cannot be re-split
per participant, and a plotted TF map cannot be correlated against anything. Keeping
the recovered components on disk turns every question below into a load.

**The layout this notebook reads.** Placebo and Psilocybin were pooled on the
**subject** axis
([`ConditionVariants.JOINED`](../../src/definitions/fields.py)), so:

```
row 0 .. P-1    Placebo      recordings      \
row P .. 2P-1   Psilocybin   recordings      /  one IVA dataset each
```

Every *recording* is one dataset, so **a participant occupies two rows** and their two
recordings got their own mixing matrices. Two consequences shape everything below:

- The channel topographies are **per recording**, so a topography contrast between the
  conditions is meaningful — that is what this variant buys.
- A bare participant label addresses **two** rows, so the row bookkeeping is not
  optional. `results.row(participant, condition)` resolves it and refuses to guess.

The sibling notebook
[`iva_component_analysis_joined_tracks.ipynb`](iva_component_analysis_joined_tracks.ipynb)
does the same for the **time**-concatenated join, where each participant is one row
carrying both tracks and the topography is shared.

**What is on disk.** From
[`src/io/iva_store.py`](../../src/io/iva_store.py): the sign-aligned per-recording TF
maps `(recordings, components, freqs, times)`, the channel topographies
`(recordings, components, channels)`, the frequency / time / channel axes, the
participant and condition of every row, the sign-alignment diagnostics, and the
per-condition stimulus onsets.

**Prerequisite.** A stored entry for the settings in the config cell. If the load
fails, run:

```bash
python scripts/run_iva_condition_comparison.py \
    --experiment assr --n_pca 10 --reuse_wavelets --store_components
```

## Configuration

In [ ]:
# ── Which stored decomposition to analyse ──────────────────────
# These five values are exactly what the store filename encodes, so they have to match
# the run that wrote it. Step 1's listing prints every entry if the settings are not
# remembered.
EXPERIMENT_NAME = ExperimentNames.ASSR
# The subject-axis join: one row per (participant, condition).
CONDITION = ConditionVariants.JOINED
VARIANT = IvaVariants.CHANNEL_JOINED
if EXPERIMENT_NAME == ExperimentNames.ASSR:
    # ASSR has no music dimension; uses a single placeholder "music type".
    MUSIC_TYPE = MusicTypeVariants.ASSR
else:
    MUSIC_TYPE = MusicTypeVariants.CLASSICAL
# The band the run was restricted to, or None for a broadband run.
BAND: str | None = None
# The run's --n_pca, which is also its component count.
N_COMPONENTS_PCA = 5

# Processed-data root the store is resolved against. None = the project's
# data/processed, which is where the CLI writes by default.
STORE_ROOT: Path | None = None

# What resolved the per-recording sign, named on every figure AND written into the
# stored trials' metadata as `sign_alignment`, so a reader always knows which
# orientation convention they are looking at.
#
# The decomposition CLI anchors the sign to the ASSR electrode topography before
# storing, and ALIGN_POLARITY_TO_MASK below re-applies the SAME anchor here — a no-op on
# a store that already carries it, a correction on one written before it existed. This
# variant has one topography per RECORDING, so the anchor is resolved per
# (participant, condition, component).
ALIGNMENT_NOTE = "corr(topography, ASSR electrode mask), per (recording, component)"

# ── Which components to draw ──────────────────────────────────
# None = every stored component. A list is 0-based, matching the IC <k+1> labels.
COMPONENTS_TO_PLOT: list[int] | None = None

# ── Figure output ─────────────────────────────────────────────
SAVE_PLOTS = True
# Per-participant grids are one figure PER COMPONENT and there can be many of them,
# so they are opt-in rather than part of a routine pass through the notebook.
WRITE_PARTICIPANT_GRIDS = False

# ── Reference lines on the TF panels ──────────────────────────
# The ASSR is continuous 40 Hz stimulation, so the stimulation frequency is the row
# worth locating on every map.
TF_FREQ_MARKS: list[float] = [iva_quality.ASSR_FREQ]
MARK_STIMULUS_ONSETS_ON_TF = True

# ── The window the paired contrast summarises ─────────────────
# A TF map is (frequency x time) per component; a paired test needs ONE number per
# (participant, component). This window is that reduction: the mean of the stored,
# sign-aligned source over a frequency band and a time span. Because the sources are
# z-scored along time before the decomposition, a positive mean reads as "more power
# in this band than this recording's own average", not as absolute power.
#
# Default: a narrow band around the ASSR stimulation frequency, whole time axis.
# Set CONTRAST_BAND to a FrequencyBandNames value instead to use a standard band.
CONTRAST_FREQ_RANGE: tuple[float, float] = (
    iva_quality.ASSR_FREQ - 2.0,
    iva_quality.ASSR_FREQ + 2.0,
)
CONTRAST_BAND: str | None = None  # e.g. "alpha"; overrides CONTRAST_FREQ_RANGE
CONTRAST_TIME_RANGE: tuple[float, float] | None = None  # None = the whole time axis

# ── Stimulus-locked epoch ─────────────────────────────────────
# Taken from src.definitions.constants.AssrEpoch so this notebook cuts the SAME epoch
# as every other onset-locked ASSR analysis: a short pre-onset baseline, then the
# stimulus plus an equally long post-stimulus interval. iva_quality.onset_window caps
# the post-onset span by the shortest inter-onset gap, so an epoch can never reach the
# next stimulus.
MIN_ONSETS_FOR_EPOCH_AVERAGE = 5

# ── An alignment weaker than this is called out ───────────────
# PC1's share of the ensemble power, from the stored sign alignment. Below this there
# is no single dominant shared map, so the group mean of that component — and any
# contrast built on it — is weak evidence.
WEAK_ALIGNMENT_EVR = 0.5

# ══ The cluster-based condition contrast (Steps 2-4) ═══════════
# DRAFT. Read the caveats in Step 2 and Step 3 before believing any number this
# produces, and read the caveat about the DATA in the Step 2 header before believing
# any of it at all.

# Contrast direction, fixed here rather than after seeing a map: everything downstream
# is CONTRAST_CONDITIONS[0] minus CONTRAST_CONDITIONS[1].
CONTRAST_CONDITIONS = (ConditionVariants.PLACEBO, ConditionVariants.PSILOCYBIN)

# Which reduction of the stored (F, T) map the test runs on. The stored map is
# 50 frequencies x 46976 samples per recording — 2.3 M cells — so testing it as it
# stands is neither tractable nor the question anyone is asking. This is a choice of
# QUESTION, not a performance knob:
#   "onset_epoch"      the onset-averaged AssrEpoch window. The stimulus-locked
#                      question — does the driven response differ between conditions —
#                      cut on the same window every other onset-locked ASSR figure
#                      uses. Averaging over the onsets cancels whatever is not
#                      time-locked to a stimulus.
#   "whole_recording"  the whole recording, averaged into TF_BIN_S-second bins. The
#                      sustained question — ongoing band power across the session —
#                      which the onset average deliberately removes.
TF_UNIT = "onset_epoch"
TF_BIN_S = 1.0  # bin width in seconds; used only by TF_UNIT == "whole_recording"

# Where the stimulus onsets come from. NOT the stored decomposition — Step 1 prints
# "Onsets stored for: none" for this run — but the sidecar written next to each
# condition's concatenated array. That array is what the wavelet transform ran on, so
# its sample indices address the very time axis the stored maps are on.
CONCATENATED_DIR: Path = (
    (STORE_ROOT or ProjectPaths.PROCESSED_DATA_DIR)
    / EXPERIMENT_NAME.value
    / PreprocessedDataVariants.CONCATENATED.value
)

# The cluster-FORMING threshold, as a two-sided Student-t critical value at this alpha
# with (participants - 1) degrees of freedom. It is NOT a significance level: it only
# decides which cells are candidates to be joined into a cluster, and moving it trades
# sensitivity to small-but-strong effects against sensitivity to broad-but-weak ones.
# Fix it in advance — choosing it after seeing the t-maps invalidates the test.
CLUSTER_FORMING_ALPHA = 0.05
# Alpha the cluster-LEVEL p-values are read against.
CLUSTER_ALPHA = 0.05
# 0 = two-sided. Nothing predicts the direction of a redistribution over the TF plane,
# and the components' polarity is arbitrary anyway (see Step 2).
CLUSTER_TAIL = 0
# None = exhaustive. The design is paired, so under the null each participant's
# difference map is equally likely to carry either sign; with P participants there are
# exactly 2**P sign-flips and MNE enumerates every one, making the p exact. That also
# sets a hard floor of 1/2**P on any attainable cluster p.
N_PERMUTATIONS: int | None = None
CLUSTER_SEED = 42
# Clusters at or below this p are listed individually in the Step 3 table. Well above
# CLUSTER_ALPHA on purpose: on a draft it is more informative to see how far the best
# cluster is from significance than to see an empty table.
CLUSTER_REPORT_MAX_P = 0.25

# ── Plots directory ───────────────────────────────────────────
# Canonical notebook layout. The analysis type carries a "_stored" suffix so these
# figures never overwrite the ones the decomposition notebook writes from its own
# in-memory results — same components, different provenance, worth keeping apart.
SPECTRUM_DIR = (
    SpectrumTypeVariants.BROADBAND.value
    if BAND is None
    else SpectrumTypeVariants.BANDS.value
)
ANALYSIS_TYPE = f"{VARIANT.value}_stored"
PLOTS_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR
    / "06-iva-condition-comparison"
    / "plots"
    / EXPERIMENT_NAME.value
    / SPECTRUM_DIR
    / ANALYSIS_TYPE
    / f"pca_{N_COMPONENTS_PCA}"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
# Band runs share the directory with nothing, but keep the prefix so a figure filename
# still says which spectrum it came from.
PLOT_PREFIX = "" if BAND is None else f"{BAND}_"

print(f"Experiment      : {EXPERIMENT_NAME.value}")
print(f"Stored run      : {VARIANT.value} / {CONDITION.value} / {MUSIC_TYPE.value}")
print(f"Spectrum        : {BAND or SpectrumTypeVariants.BROADBAND.value}")
print(f"Components      : {N_COMPONENTS_PCA}  (the run's --n_pca)")
print(f"Store root      : {STORE_ROOT or ProjectPaths.PROCESSED_DATA_DIR}")
print(f"Plots directory : {PLOTS_DIR}  (saving: {SAVE_PLOTS})")
print(
    f"Cluster test    : {CONTRAST_CONDITIONS[0].value} - "
    f"{CONTRAST_CONDITIONS[1].value} on the {TF_UNIT!r} unit"
)

## Data Loading — read the stored decomposition

Two calls. `list_iva_results` enumerates what the experiment has on disk, which is the
quickest way to see which runs exist and what settings they used — the filename carries
the variant, the music type, the spectrum and `n_pca`. `load_iva_components` then reads
one entry, either by path or by the run descriptor as here.

An entry is one decomposition. A `--n_pca` or band sweep leaves one file per setting,
so nothing below can silently mix two runs.

In [ ]:
available = list_iva_results(EXPERIMENT_NAME, processed_data_dir=STORE_ROOT)
print(f"Stored IVA results for {EXPERIMENT_NAME.value} ({len(available)}):")
for entry in available:
    print(f"  {entry.parent.name:<14} {entry.name}")
if not available:
    print(
        "  (nothing stored yet — run the CLI with --store_components first:\n"
        f"   python scripts/run_iva_condition_comparison.py --experiment "
        f"{EXPERIMENT_NAME.value} --n_pca {N_COMPONENTS_PCA} --reuse_wavelets "
        "--store_components)"
    )

results = load_iva_components(
    experiment=EXPERIMENT_NAME,
    condition=CONDITION,
    variant=VARIANT,
    music_type=MUSIC_TYPE,
    band=BAND,
    n_pca=N_COMPONENTS_PCA,
    processed_data_dir=STORE_ROOT,
)
print(f"\nLoaded {results.path}")

## Dataset Selection — unpack the arrays and the row bookkeeping

One stored entry, so this cell just gives its contents the names the rest of the
notebook uses, and pulls out the three things that turn arrays back into an analysis:
the **row bookkeeping** (participant and condition per row), the **axes** (frequency,
time, channels), and the **onsets**.

`results.topo_info()` rebuilds an MNE `Info` from the stored channel names and applies
the project montage — that is why the names were stored, since a channel pattern is
only a topography once its values sit on a scalp.

In [ ]:
LABEL = results.label  # the canonical "Joined_<MusicType>" product name

tf_maps = results.tf_maps  # (S, K, F, T) per-recording component TF maps
channel_patterns = results.channel_patterns  # (S, K, C) per-recording topographies
freqs = results.freqs  # (F,) Hz
times = results.times  # (T,) s
sfreq = results.sfreq

n_subjects, n_components, n_freqs, n_times = tf_maps.shape
n_channels = results.n_channels

# The row bookkeeping. Every array above is indexed (recording, component, ...), so
# these are what turn a row index back into a person under a condition — and, here,
# what keeps a participant's two rows apart.
subject_participants = list(results.participants)
subject_conditions = list(results.subject_conditions)
# Condition order as the run laid the subject axis out: one whole block each.
condition_rows = list(dict.fromkeys(subject_conditions))
participants = sorted(set(subject_participants))
# Participants contributing every condition — the ones a paired read-out can use.
paired_participants = [
    p for p in participants if all(results.rows(p, c) for c in condition_rows)
]

comp_indices = (
    list(range(n_components))
    if COMPONENTS_TO_PLOT is None
    else list(COMPONENTS_TO_PLOT)
)
out_of_range = [k + 1 for k in comp_indices if not 0 <= k < n_components]
if out_of_range:
    raise ValueError(
        f"COMPONENTS_TO_PLOT names IC {out_of_range}, outside 1..{n_components}."
    )

# Stimulus onsets, on the one shared time axis (the conditions were aligned together).
# ``None`` for an experiment without stimulus annotations.
onsets_by_condition = {c: results.stimulus_onsets(c) for c in condition_rows}
reference_onsets = onsets_by_condition.get(condition_rows[0])
onset_times = (
    reference_onsets / sfreq
    if MARK_STIMULUS_ONSETS_ON_TF and reference_onsets is not None
    else np.array([])
)

# The topomap layout, rebuilt from the stored channel names.
topo_info = results.topo_info()

print(f"Product     : {LABEL}")
print(f"TF maps     : {tf_maps.shape}  (recordings x components x freqs x times)")
print(f"Topographies: {channel_patterns.shape}  (recordings x components x channels)")
print(f"Time axis   : {times[-1]:.1f} s @ {sfreq} Hz")
print(f"Freq axis   : {freqs[0]:.1f}-{freqs[-1]:.1f} Hz ({n_freqs} bins)")
print(f"Conditions  : {condition_rows}")
print(
    f"Participants: {len(participants)} ({len(paired_participants)} with every "
    "condition)"
)
print(f"Components  : showing {len(comp_indices)} of {n_components}")
print(f"Onsets      : {0 if reference_onsets is None else len(reference_onsets)}")

# ── Small helpers used by several steps below ─────────────────


def _grid(n_panels: int, width: float = 3.6, height: float = 2.9):
    """A subplot grid wide enough for *n_panels*, at most 5 columns."""
    ncols = min(5, max(1, n_panels))
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(width * ncols, height * nrows), squeeze=False
    )
    for j in range(n_panels, nrows * ncols):
        axes[j // ncols][j % ncols].axis("off")
    return fig, axes, ncols


def _save(fig, name: str) -> None:
    """Write *fig* into PLOTS_DIR under the band prefix, when saving is on."""
    if not SAVE_PLOTS:
        return
    path = PLOTS_DIR / f"{PLOT_PREFIX}{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"saved {path}")


def _benjamini_hochberg(pvals) -> np.ndarray:
    """BH step-up adjusted p-values; ``NaN`` entries pass through untouched.

    Written out rather than imported so the notebook has no statsmodels dependency.
    """
    p = np.asarray(pvals, dtype=float)
    adjusted = np.full(p.shape, np.nan)
    finite = ~np.isnan(p)
    if not finite.any():
        return adjusted
    vals = p[finite]
    order = np.argsort(vals)
    n = vals.size
    stepped = vals[order] * n / np.arange(1, n + 1)
    # Enforce monotonicity from the largest p downwards.
    stepped = np.minimum.accumulate(stepped[::-1])[::-1]
    out = np.empty(n)
    out[order] = np.clip(stepped, 0.0, 1.0)
    adjusted[finite] = out
    return adjusted


def _contrast_masks():
    """``(freq_mask, time_mask, description)`` for the paired-contrast window."""
    if CONTRAST_BAND is not None:
        low, high = FREQUENCY_BANDS[CONTRAST_BAND]
        band_name = CONTRAST_BAND
    else:
        low, high = CONTRAST_FREQ_RANGE
        band_name = f"{low:.1f}-{high:.1f} Hz"
    freq_mask = (freqs >= low) & (freqs <= high)
    if not freq_mask.any():
        raise ValueError(
            f"No stored frequency falls in [{low}, {high}] Hz; the stored grid is "
            f"{freqs[0]:.1f}-{freqs[-1]:.1f} Hz. Adjust CONTRAST_FREQ_RANGE / "
            "CONTRAST_BAND, or load a run whose band covers it."
        )
    return freq_mask, band_name


def _time_mask(axis_times: np.ndarray) -> np.ndarray:
    """Boolean mask over *axis_times* for CONTRAST_TIME_RANGE (None = everything)."""
    if CONTRAST_TIME_RANGE is None:
        return np.ones(axis_times.size, dtype=bool)
    start, stop = CONTRAST_TIME_RANGE
    mask = (axis_times >= start) & (axis_times <= stop)
    if not mask.any():
        raise ValueError(
            f"No stored sample falls in [{start}, {stop}] s; the axis spans "
            f"0-{axis_times[-1]:.1f} s. Adjust CONTRAST_TIME_RANGE."
        )
    return mask

## Configuration — trials / SNR workflow

The settings the ported Steps 2-12 use. Read after the store is loaded so the z-score mode can be taken off the file.

In [ ]:
# ══ Trials / SNR workflow configuration ═══════════════════════
# Ported from the time-concatenated sibling; the only structural difference in this
# variant is that the topography (and thus the spatial filter and the PCA subspace) is
# estimated PER RECORDING, so every projection below is condition-specific. Read after
# the store is loaded, so the z-score mode can be taken off the file.
zscore_mode = results.extras.get("zscore_mode")

# ── Raw wavelet tracks for the spatial-filter projection ──────
# Read from the notebook SUBSET caches and nothing else. Deliberately not through
# scripts.notebook_helpers.load_paired_condition_wavelets: that calls load_analyzers
# first, which loads the whole concatenated preprocessed recording for the cohort
# before any wavelet is touched, and then reads the source-of-truth wavelet cache —
# ~52 GB per condition for ASSR. Neither finishes in a notebook.
#
# The subset caches are the per-extent copies under the stage-03 notebook, written
# UNCOMPRESSED precisely so they are cheap to read back. Nothing here recomputes a
# wavelet, and nothing reads data/processed/<experiment>/wavelets.
WAVELET_SUBSET_CACHE_DIR: Path = (
    resolve_notebook_wavelet_cache_dir(EXPERIMENT_NAME)
    / SpectrumTypeVariants.BROADBAND.value
)
# Time-axis segment order, so filtered_placebo comes first and filtered_psilocybin
# second.
CONDITIONS_TO_POOL = list(REAL_CONDITIONS)
# The frequency grid the cache was written with; it is part of the cache filename.
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)
# Needed only to read the participant metadata (a CSV parse and a directory listing —
# no EEG is loaded), which is what names the rows of each cache's subject axis.
COORDINATE_SYSTEM = CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]

# Trim the cached time axis further after loading. None keeps the cached extent.
# The CHANNEL axis is not a knob: the spatial filters are (components x channels) of
# the stored run, so only a cache with exactly that many channels can be used, and the
# selection below refuses anything else rather than silently truncating a filter.
N_TIMES_SUBSET: int | None = None
# Keep only the first N participants of the stored cohort. Applied to the projection,
# so it does lower the peak here.
N_PAIRS_SUBSET: int | None = None

print(f"Wavelet subset cache : {WAVELET_SUBSET_CACHE_DIR}")
print(f"Track trim           : n_times={N_TIMES_SUBSET}, n_pairs={N_PAIRS_SUBSET}")

# ── The binary ASSR-electrode filter (Step 5) ─────────────────
# The include-list in config/assr_electrodes/<coordinate system>.csv — the standard
# fronto-central selection the 40 Hz steady-state response is read from — used as a
# second spatial filter alongside the IVA ones.
#
# True averages over the selected electrodes; False sums them. The mean is the default
# because it keeps the output in the same wavelet-power units as the input and does not
# scale with how many electrodes the list happens to contain, which a raw binary sum
# does.
ASSR_MASK_NORMALIZE = True
# Preprocessing drops the boundary electrodes, so a listed electrode can be absent from
# a recording. True refuses to under-select rather than quietly averaging over fewer
# electrodes than the list names; False accepts the intersection and logs what was
# missing.
ASSR_MASK_STRICT = True

# ── The 40 Hz trial extraction (Steps 6-8) ────────────────────
# The frequency selections the per-trial time courses are read at, as
# (label, centre Hz, half-width Hz). A zero half-width takes the single nearest bin —
# on this 1 Hz grid that IS the 40 Hz row. A positive one averages every bin inside the
# closed interval, which is what survives wavelet smearing and a few Hz of stimulator
# drift. Both are extracted because they answer the same question with different
# exposure to that smearing, and the extracted arrays are small enough that keeping
# both costs nothing.
FREQ_SELECTIONS: list[tuple[str, float, float]] = [
    ("40hz", iva_quality.ASSR_FREQ, 0.0),
    (
        f"{iva_quality.ASSR_FREQ - iva_quality.TF_ANCHOR_HALFWIDTH_HZ:g}"
        f"-{iva_quality.ASSR_FREQ + iva_quality.TF_ANCHOR_HALFWIDTH_HZ:g}hz",
        iva_quality.ASSR_FREQ,
        iva_quality.TF_ANCHOR_HALFWIDTH_HZ,
    ),
]

# Labels of the two binary ASSR-electrode REFERENCE rows, alongside the learned
# "IC <k>" rows. Both the fixed masks and the learned filters share ONE source axis in
# Step 6, so a trial-level analysis can slice "the same trial under each filter" instead
# of joining differently-shaped arrays; the labels are what tell them apart afterwards.
#
# Two references, not one:
#   * "ASSR-mask (full)" — the fixed fronto-central electrode average on the WHOLE raw
#     wavelet data (every channel the recording kept).
#   * "ASSR-mask (PCA)"  — the SAME electrode average, but on the raw wavelet data first
#     projected onto each participant's per-recording PCA subspace: the very subspace
#     the IVA components were estimated in (Step 5 recovers it from the stored patterns).
#     It is the apples-to-apples reference for the ICs — it can only see what the PCA
#     reduction kept, exactly as the components can; the "(full)" row shows what that
#     reduction discarded. Unlike "(full)", the PCA row is a SIGNED combination of
#     channels, so it behaves like an IC under the per-trial normalisation (Step 8).
BINARY_FILTER_LABEL = "ASSR-mask (full)"
BINARY_FILTER_PCA_LABEL = "ASSR-mask (PCA)"
#   * "ASSR-mask (PCA, z)" — the same subspace-projected electrode average, but read off
#     the Z-SCORED wavelet rather than the raw one: literally the signal the IVA was
#     handed, after the channel PCA and before the unmixing. It is the reference-side
#     analogue of the "stored_prestim" IC rows, and it fills the same missing cell:
#
#                        | raw wavelet          | z-scored wavelet
#       -----------------+----------------------+---------------------
#       no baseline      | --                   | "zscored" variant
#       per-trial base   | "ASSR-mask (PCA)"    | "ASSR-mask (PCA, z)"
#
#     Reading it beside "ASSR-mask (PCA)" separates the two things the IVA's input
#     preparation does to the ROI: the channel reduction (already in the "(PCA)" row)
#     and the per-(channel, frequency) rescaling that z-scoring applies, which changes
#     the effective electrode weighting rather than just its overall scale.
#
#     It needs NO second normalisation. Unlike the equal-weight ROI row of Step 8 — a
#     mean of many normalised channels, whose own baseline SD is well below 1 — this is
#     a single weighted signal, so the per-trial baseline leaves it at exactly 1 by
#     construction, in the same units as every IC row.
BINARY_FILTER_PCA_Z_LABEL = "ASSR-mask (PCA, z)"
# The reference rows, in the order they are appended after the ICs on the source axis.
MASK_LABELS = [
    BINARY_FILTER_LABEL,
    BINARY_FILTER_PCA_LABEL,
    BINARY_FILTER_PCA_Z_LABEL,
]

# Where the stimulus onsets come from. NOT the stored decomposition — Step 1 prints
# "Onsets stored for: none" for this run — but the sidecar written next to each
# condition's concatenated array. That array is what the wavelet transform was run on,
# so its sample indices address the very time axis the projected tracks are on.
CONCATENATED_DIR: Path = (
    ProjectPaths.PROCESSED_DATA_DIR
    / EXPERIMENT_NAME.value
    / PreprocessedDataVariants.CONCATENATED.value
)

# Where the extracted trials are written: under data/processed (gitignored), beside the
# component store the spatial filters came from.
SAVE_TRIALS = True
TRIALS_DIR: Path = (
    (STORE_ROOT or ProjectPaths.PROCESSED_DATA_DIR)
    / EXPERIMENT_NAME.value
    / "assr_trials"
)

print(f"Freq selections : {[name for name, _c, _h in FREQ_SELECTIONS]}")
print(f"Onsets from     : {CONCATENATED_DIR}")
print(f"Trials directory: {TRIALS_DIR}  (saving: {SAVE_TRIALS})")

# ── The participant-level tests (Step 10-11) ──────────────────
# Which frequency selection the tests reduce over. Any label present in FREQ_SELECTIONS
# is valid: "40hz" reads the single 40 Hz bin, "35-45hz" averages the band (more robust
# to wavelet smearing and stimulator drift).
TEST_SELECTION = "40hz"

# A fully custom frequency window for the test, as (centre_hz, halfwidth_hz). When set it
# is appended to FREQ_SELECTIONS (so Step 6 extracts it like the rest) and selected as
# the test frequency, overriding TEST_SELECTION. None = use TEST_SELECTION above.
#   (40.0, 5.0) -> mean of 35-45 Hz;   (40.0, 0.0) -> the single 40 Hz bin.
TEST_FREQ_WINDOW: tuple[float, float] | None = None
if TEST_FREQ_WINDOW is not None:
    _c, _hw = TEST_FREQ_WINDOW
    TEST_SELECTION = f"test_{_c - _hw:g}-{_c + _hw:g}hz" if _hw else f"test_{_c:g}hz"
    if TEST_SELECTION not in {name for name, _cc, _hh in FREQ_SELECTIONS}:
        FREQ_SELECTIONS.append((TEST_SELECTION, _c, _hw))

# The stimulus window the test reads, in seconds. None = the paradigm's full driven
# interval (0 - AssrEpoch.STIMULUS_DURATION_S, i.e. 0-500 ms). A (start, stop) tuple
# overrides it, e.g. (0.2, 0.5) to skip the onset transient and read only the sustained
# steady state.
TEST_STIMULUS_INTERVAL: tuple[float, float] | None = None

# The signal-normalisation variants the whole test + figure suite runs on, side by side
# (Step 10-12). They differ in TWO independent ways — which representation the LEARNED
# rows come from, and how the signal is put into comparable units — and the three of
# them fill the cells of that 2x2 that are actually reachable:
#
#                   | no per-trial baseline | per-trial pre-stimulus baseline
#   ----------------+-----------------------+--------------------------------
#   re-projected    |          --           | "prestim"
#   stored sources  | "zscored"             | "stored_prestim"
#
# Only the LEARNED rows move between those cells. Both ASSR-mask references are the
# fixed quantity every variant is measured against, so in "stored_prestim" they are the
# "prestim" rows verbatim — the raw wavelet power converted by the mask and referenced
# to each trial's own pre-stimulus window. What changes between "prestim" and
# "stored_prestim" is how the COMPONENT is read, and nothing else.
#
#   "zscored"        — the wavelet power is z-scored along time FIRST (the
#                      standardisation the IVA decomposition itself uses), then
#                      converted by the topomap. The IVA rows are the STORED z-scored
#                      sources (full recording, all fitting onsets); the ASSR-mask rows
#                      are the mask applied to the z-scored 12 s subset. No per-trial
#                      baseline — the z-score IS the normalisation.
#   "prestim"        — the RAW wavelet power is converted by the topomap (Step 4), then
#                      every trial is referenced to its own pre-stimulus baseline
#                      (Step 8). Units of pre-stimulus SD, 12 s subset, ~9 trials for
#                      every row.
#   "stored_prestim" — the IC rows are the STORED IVA sources, cut into trials and
#                      referenced to their own pre-stimulus baseline (Step 9b): the same
#                      units and the same measure as "prestim", but read off the
#                      decomposition instead of re-derived from the caches, so ~148
#                      trials instead of ~9. The ASSR-mask rows are "prestim"'s,
#                      unchanged — raw wavelet, 12 s subset, ~9 trials.
#
# Running all three answers two separate questions: "does the normalisation change the
# conclusion?" ("zscored" vs the other two) and "does reading the sources off the
# decomposition change it?" ("prestim" vs "stored_prestim"). Because the reference rows
# are held fixed, the second question is asked cleanly: a "prestim" IC row and a
# "stored_prestim" IC row are compared against exactly the same numbers.
#
# Caveat, and it applies to "zscored" and "stored_prestim" alike: their IC rows span the
# full 188 s recording (~148 trials) while their mask rows span the 12 s subset (~9), so
# an IC row's per-participant value is the better-averaged of the two. That is a
# difference in how much noise each side carries, not in units — both are already in
# pre-stimulus SD — but it does mean an IC-vs-reference gap (4c, and the Step 12 SNR
# comparison) should not be read as a pure statement about the spatial filters. Set
# STORED_PRESTIM_ONSETS = "subset" below to remove it. "prestim" alone has every row on
# identical footing. The CLI counterpart, scripts/run_assr_snr_grid.py, streams the FULL
# cache and so has no such asymmetry in any variant.
TEST_VARIANTS = ["zscored", "prestim", "stored_prestim"]

# Which onsets the "stored_prestim" IC rows are cut at. The ASSR-mask rows are always
# "prestim"'s, so they are always on the 12 s subset whatever this is set to.
#   "all"    — every onset the recording carries (~148 per condition). The reason the
#              variant is worth running: 16x the trials of the cached 12 s subset, at no
#              extra I/O, because the stored sources already span the whole recording.
#   "subset" — only the onsets inside the 12 s cache extent (~9), matching "prestim"
#              trial for trial. Use it to isolate the ONE thing the two variants differ
#              in — stored sources vs re-projected — with the trial count held fixed,
#              which also puts the IC and reference rows back on equal footing.
STORED_PRESTIM_ONSETS = "all"

# How a trial's 275-sample time course becomes one number.
#   "stimulus"            mean over the driven interval.
#   "stimulus_minus_rest" that minus the mean over the rest of the epoch, which cancels
#                         whatever offset the per-trial normalisation left behind,
#                         because both halves carry it equally.
# Fix this BEFORE looking at any p-value.
RESPONSE_MEASURE = "stimulus"

# Anchor each participant's component polarity to the ASSR electrodes before testing.
# The run's own sign alignment (PC1 of the TF maps) makes a component's sign consistent
# in the sense PC1 defines, which is NOT "positive means more fronto-central power" —
# measured on this cohort, the sign of a component's pattern weight over the mask
# electrodes splits across participants on every IC. Re-anchoring to the mask makes a
# higher value mean more 40 Hz power over that area for EVERY participant, which is
# what lets the directional prior below be stated at all.
#
# Why this particular anchor is legitimate and a data-driven one is not: it reads only
# the fixed spatial reference (each recording's pattern projected onto the ASSR
# electrodes), never the tested response, so it cannot manufacture a condition
# difference. A flip taken from the tested quantity instead — e.g. "make Placebo
# positive" — breaks the exchangeability the paired test rests on and inflates the
# one-sided false-positive rate from 0.05 to ~0.68 under a simulated null.
ALIGN_POLARITY_TO_MASK = True

# ── How the binary ROI reference row is built (Step 8) ────────
# True  — each ROI electrode is referenced to its own pre-stimulus MEAN per trial and
#         divided by its baseline SD POOLED over trials, BEFORE the ROI is averaged; the
#         average is then referenced to its own pre-stimulus again so the row lands back
#         in units of its own baseline SD.
#
#         The divisor is pooled deliberately. A 25-sample baseline SD varies ~4x more
#         than sampling theory predicts, so a channel that happens to be quiet before
#         one onset gets divided by a spuriously small number — which inflates its WHOLE
#         epoch, turning any slow post-stimulus drift into a large sustained deviation.
#         Since the ROI is a mean, those channels dominate. Left per-trial the ROI course
#         peaks ~6x too high and plateaus at ~46% of peak instead of returning to
#         baseline; pooled, it returns (-16%) and matches the raw signal, the
#         power-weighted row and any single electrode. The MEAN stays per trial, so the
#         drift correction — the part that genuinely has to be local — is unaffected.
# False — the historical order: average the raw power over the ROI, normalise the mean
#         afterwards.
#
# Averaging raw wavelet POWER first leaves every electrode weighted by its own power
# level, which measured on this cohort is uncorrelated (r = -0.10) with whether that
# electrode carries any 40 Hz response. The mask's intent is equal weight, so the
# normalisation has to come first. The usual defence of averaging first — coherent
# signal adds, noise cancels, sqrt(N) — is about VOLTAGE and does not transfer to power,
# where there is no phase to cancel.
#
# The second normalisation is not redundant: one normalised channel has baseline SD 1,
# but the MEAN of correlated channels does not (~0.39 here, ~6.5 effectively independent
# electrodes of 35). Without it the ROI row sits ~2.5x above every IC row, which IS in
# units of its own baseline SD, and Step 12 compares the two directly.
#
# Applies to the BINARY mask only. "ASSR-mask (PCA)" has deliberately unequal signed
# weights, so "equal weight per electrode" is not what that row is for.
ROI_CHANNELWISE = True

# Test direction for the 4b contrast, computed as Placebo - Psilocybin. The prior is
# that psilocybin LOWERS the 40 Hz response over the fronto-central area, i.e.
# Placebo > Psilocybin, i.e. a positive difference: "greater". This is only meaningful
# because ALIGN_POLARITY_TO_MASK has made "higher = more power there" true of every row.
# One-sided halves the attainable p (floor 1/2**P rather than 2/2**P) and forfeits any
# claim if the effect runs the other way; it is legitimate only because the direction
# was fixed in advance.
#
# 4c stays two-sided regardless — nothing predicts whether a learned component should
# beat a fixed electrode selection.
CONTRAST_ALTERNATIVE = "greater"

print(f"Tests on      : {TEST_SELECTION}, response = {RESPONSE_MEASURE}")

# ── Summary figures (Step 11) ─────────────────────────────────
# One colour per condition, used by every panel so a line never has to be looked up.
CONDITION_COLORS = {
    ConditionVariants.PLACEBO.value: "#0F6E8C",
    ConditionVariants.PSILOCYBIN.value: "#A6357F",
}
# Spread drawn around each mean time course, ACROSS PARTICIPANTS (never across trials —
# trials within a participant are correlated, so their spread understates the real
# uncertainty). "sem" is mean +/- standard error, "iqr" the 25-75 band around the median.
COURSE_SPREAD = "sem"
# Percentile bootstrap over participants for the forest intervals. The Wilcoxon p is
# exact and does not come from this; the interval is only there to show the spread.
N_BOOTSTRAP = 10_000
BOOTSTRAP_SEED = 42
ALPHA = 0.05


---
## Step 2 — Recover the IVA spatial filters (one per recording)

The file stores the forward patterns (topographies), not the filters, and the two are not
interchangeable. For recording *s* the filter `U_s = W_s @ P_s` (components x channels) is
the *backward* operator that extracts a source from the channels — what is needed to
project a new signal — while the stored pattern `A_s = pinv(U_s)` is the *forward* model
that belongs on a topomap. After `iva_g(..., whiten=True)` the two are nearly unrelated
(Haufe et al., 2014), so using the pattern here would be the classic filter-vs-pattern
error.

The filter inverts the stored pattern exactly, and the round trip `U_s @ A_s = I` is
checked rather than trusted. **This variant estimates one mixing matrix PER RECORDING**, so
— unlike the time-concatenated sibling, where one filter is shared across both tracks — a
participant has a *separate* filter under each condition. Every downstream projection
therefore picks the filter of the specific (participant, condition), which `store_row`
(Step 4) resolves via `results.row`.


In [ ]:
# channel_patterns[s] is (K, C) and holds A_s transposed, so A_s is its transpose.
# Inverting that gives back the (K, C) spatial filter U_s of recording s. There are
# n_subjects = 2P of them: one per (participant, condition), NOT one per participant.
spatial_filters = np.stack(
    [np.linalg.pinv(channel_patterns[s].T) for s in range(n_subjects)]
)  # (S, K, C) per recording

# Check the round trip instead of assuming it: U_p @ A_p must be the identity. Expect a
# residual around 1e-7, not 1e-15: the store keeps the patterns as float32 by default, so
# the inversion inherits float32 precision. Orders of magnitude above that would mean the
# stored patterns are rank-deficient and the recovered filter is not the operator the run
# actually used.
identity_error = np.array(
    [
        np.abs(spatial_filters[s] @ channel_patterns[s].T - np.eye(n_components)).max()
        for s in range(n_subjects)
    ]
)

print(
    f"Spatial filters : {spatial_filters.shape}  (recordings x components x channels)"
)
print(
    f"|U_p A_p - I|   : max {identity_error.max():.2e} "
    f"(worst recording {subject_participants[int(identity_error.argmax())]}/"
    f"{subject_conditions[int(identity_error.argmax())]})"
)
if identity_error.max() > 1e-6:
    raise ValueError(
        f"The recovered filters do not invert the stored patterns (max residual "
        f"{identity_error.max():.2e}). The stored patterns are probably "
        "rank-deficient, so the filter cannot be recovered from them."
    )

---
## Step 3 — Read the raw, un-z-scored wavelet tracks from the subset caches

**Caches only.** This step reads the per-extent subset caches under
`notebooks/03-wavelet-analysis/wavelet_cache/` and nothing else: no wavelet is
recomputed, the 52 GB source-of-truth caches under
`data/processed/<experiment>/wavelets/` are never opened, and no preprocessed
recording is loaded.

That is a change from the obvious route.
`load_paired_condition_wavelets` would call `load_analyzers` first, which loads the
whole concatenated recording for the cohort *before* touching a wavelet, and then reads
the source cache — which is why it never finished. The subset caches are written
uncompressed for exactly this purpose, and they already carry everything needed to
interpret them: the array as `(subjects, channels, frequencies, times)`, the channel
names, the frequency grid and the sampling rate.

**No standardisation at any stage.** The cache holds wavelet power as it was
transformed; nothing here z-scores it, per track or jointly. The output of Step 4 is
therefore in the cache's own power units, so an overall power difference between the
conditions survives into the result instead of being normalised away. (The *filter* was
estimated on z-scored data — that is baked into the stored decomposition and cannot be
undone here. What changes is the signal it is applied to.)

**The one thing that has to match is the channel axis.** The filters are
`(components x channels)` of the stored run, so a cache with a different channel count
is unusable — dropping columns from a spatial filter does not restrict it to those
channels, it makes a different and meaningless operator. The selection below therefore
requires an exact channel-count match per condition and reports precisely what to build
if it is missing.

Participant labels come from the dataset **metadata** (a CSV parse plus a directory
listing), because a cache's subject axis is in concatenation order and carries no
labels of its own.

In [ ]:
def _npz_headers(path: Path) -> dict[str, tuple]:
    """Array shapes and dtypes inside an ``.npz``, WITHOUT decompressing anything.

    Lets the inventory below report what a cache holds at no cost; reading ``data``
    normally would pull the whole tensor into memory.
    """
    readers = {
        (1, 0): np.lib.format.read_array_header_1_0,
        (2, 0): np.lib.format.read_array_header_2_0,
    }
    out: dict[str, tuple] = {}
    with zipfile.ZipFile(path) as archive:
        for name in archive.namelist():
            if not name.endswith(".npy"):
                continue
            with archive.open(name) as handle:
                version = np.lib.format.read_magic(handle)
                shape, _fortran, dtype = readers[version](handle)
            out[name[: -len(".npy")]] = (shape, str(dtype))
    return out


def _subset_cache_entries(condition) -> list[dict]:
    """Subset-cache entries for one condition, at this frequency grid.

    Filenames are ``<label>__wavelet_<repr>__<f0>_<f1>_<n>__S<n>_C<c>_T<t>__freqdim1r``,
    written by ``scripts.notebook_helpers._save_subset_cache``. Parsed rather than
    reconstructed, because the extent is not known in advance — that is exactly what the
    inventory is for.
    """
    label = f"{condition.value}_{MUSIC_TYPE.value}"
    freq_signature = f"{FREQS[0]:.3f}_{FREQS[-1]:.3f}_{len(FREQS)}"
    pattern = f"{label}__wavelet_power__{freq_signature}__*__freqdim1r.npz"
    entries = []
    for candidate in sorted(WAVELET_SUBSET_CACHE_DIR.glob(pattern)):
        match = re.search(r"__S(\d+)_C(\d+)_T(\d+)__", candidate.name)
        if match is None:
            continue  # a "full"-extent entry; not usable without knowing its shape
        shape, dtype = _npz_headers(candidate).get("data", (None, None))
        entries.append(
            {
                "condition": condition.value,
                "path": candidate,
                "n_subjects": int(match.group(1)),
                "n_channels": int(match.group(2)),
                "n_times": int(match.group(3)),
                "shape": shape,
                "dtype": dtype,
                "size_gb": candidate.stat().st_size / 1e9,
            }
        )
    return entries


inventory = [
    entry
    for condition in CONDITIONS_TO_POOL
    for entry in _subset_cache_entries(condition)
]
print(f"Subset caches in {WAVELET_SUBSET_CACHE_DIR}:")
if not inventory:
    print("  (none for this experiment / frequency grid)")
for entry in inventory:
    usable = "USABLE" if entry["n_channels"] == n_channels else f"needs C{n_channels}"
    print(
        f"  {entry['condition']:<12} S{entry['n_subjects']:<3} "
        f"C{entry['n_channels']:<4} T{entry['n_times']:<6} "
        f"{entry['size_gb']:6.2f} GB  {entry['dtype']}  [{usable}]"
    )

# ── Pick one entry per condition: the channel count must match the run ─
selected = {}
for condition in CONDITIONS_TO_POOL:
    candidates = [
        entry
        for entry in _subset_cache_entries(condition)
        if entry["n_channels"] == n_channels
    ]
    if not candidates:
        available = [
            f"C{entry['n_channels']}_T{entry['n_times']}"
            for entry in _subset_cache_entries(condition)
        ]
        # Both flags matter: the subset cache is keyed by channels AND time, so
        # omitting --n_times writes a "full"-extent entry instead. Suggest the time
        # extent an already-usable entry uses, so the two conditions end up matched.
        usable_times = [
            entry["n_times"]
            for other in CONDITIONS_TO_POOL
            for entry in _subset_cache_entries(other)
            if entry["n_channels"] == n_channels
        ]
        suggested_times = (
            max(usable_times) if usable_times else (N_TIMES_SUBSET or 3000)
        )
        raise FileNotFoundError(
            f"No subset cache for {condition.value}_{MUSIC_TYPE.value} with "
            f"{n_channels} channel(s), which is what the stored run used. A cache with "
            "a different channel count cannot be substituted: the spatial filters are "
            "indexed by those channels.\n"
            "Build it once (it reads the source cache, so run it as a job, not here):\n"
            f"  python scripts/run_iva_condition_tracks.py --experiment "
            f"{EXPERIMENT_NAME.value} --n_pca {N_COMPONENTS_PCA} --n_channels "
            f"{n_channels} --n_times {suggested_times} --reuse_wavelets "
            f"--subset_cache\n"
            f"Available for this condition: {available or 'nothing'}"
        )
    # Longest time axis available, then trimmed by N_TIMES_SUBSET below.
    selected[condition] = max(candidates, key=lambda entry: entry["n_times"])
    print(f"\nUsing for {condition.value}: {selected[condition]['path'].name}")

# ── Participant labels for each cache's subject axis (metadata only) ───
handler = DatasetHandler(EXPERIMENT_NAME, COORDINATE_SYSTEM)
raw_participants = {}
for condition in CONDITIONS_TO_POOL:
    frame = DatasetFilter.filter_dataset_by_all_categories(
        handler.dataset_metadata,
        handler.excluded_participants_metadata,
        [MUSIC_TYPE],
        [condition],
        EXCLUSION_CATEGORIES,
    )
    raw_participants[condition.value] = [
        participant_label(pid) for pid in frame[SingleDataMetadata.PARTICIPANT_ID]
    ]

# ── Read the arrays ───────────────────────────────────────────────────
raw_tracks = {}
raw_freqs = FREQS
for condition in CONDITIONS_TO_POOL:
    entry = selected[condition]
    with np.load(entry["path"]) as cached:
        track = cached["data"]  # (S, C, F, T) — already reshaped by the writer
        cache_freqs = cached["freqs"]
        cache_sfreq = float(cached["sfreq"])
        cache_channels = (
            cached["feature_names"].tolist()
            if cached["has_feature_names"].item()
            else None
        )
    if N_TIMES_SUBSET is not None:
        track = track[..., :N_TIMES_SUBSET]

    labels = raw_participants[condition.value]
    if len(labels) != track.shape[0]:
        raise ValueError(
            f"{condition.value}: the cache has {track.shape[0]} subject(s) but the "
            f"metadata names {len(labels)}. The cache was written for a different "
            "cohort or exclusion set."
        )
    if cache_channels is not None and cache_channels != list(results.channel_names):
        raise ValueError(
            f"{condition.value}: the cache's channel axis does not match the stored "
            f"run's.\n  cache: {cache_channels[:5]}...\n"
            f"  store: {list(results.channel_names)[:5]}...\n"
            "The spatial filters are indexed by the stored channels."
        )
    if cache_sfreq != sfreq:
        raise ValueError(
            f"{condition.value}: cache sfreq {cache_sfreq} != stored {sfreq}."
        )
    if BAND is not None:
        track, raw_freqs = slice_to_band(track, cache_freqs, BAND)
    elif not np.allclose(cache_freqs, freqs):
        raise ValueError(
            f"{condition.value}: the cache spans "
            f"{cache_freqs[0]:.1f}-{cache_freqs[-1]:.1f} Hz ({cache_freqs.size} bins) "
            f"but the stored run has {freqs[0]:.1f}-{freqs[-1]:.1f} Hz "
            f"({n_freqs} bins)."
        )
    raw_tracks[condition.value] = track

print("\nRaw tracks (un-z-scored), from the subset caches:")
for name, track in raw_tracks.items():
    print(
        f"  {name:<12}: {track.shape}  (subjects x channels x freqs x times), "
        f"{track.nbytes / 1e9:.2f} GB"
    )
print(f"Channel axis  : {n_channels} channel(s), matching the stored run")
print("z-scoring     : none applied at any stage")

---
## Step 4 — Project each condition's track through its own (participant, condition) filter

For every participant and condition, the filter `U_(p,c)` of that recording is applied to
that participant's raw wavelet track for that condition:

```
sources(f, t) = U_(p,c) @ X_(p,c)(:, f, t)   # (K, C) @ (C,) -> (K,)
```

Because the filter only contracts the channel axis, frequency and time pass through
untouched. Rows are matched by participant label through `store_row` / `results.row`, never
by position, and only participants the store holds under **every** condition
(`paired_participants`) and both caches contain are projected — the unit a paired read-out
needs. The one difference from the time-concatenated sibling: the filter is
condition-specific here, because the topography is per recording.


In [ ]:
# Match by participant label, never by position: the store's row order and each
# cache's subject axis are independent, and the two conditions' cohorts differ in size.
cache_row = {
    name: {participant: i for i, participant in enumerate(labels)}
    for name, labels in raw_participants.items()
}

# This variant estimates ONE mixing matrix PER RECORDING, so a participant has a
# separate filter under each condition. Only participants the store holds under EVERY
# condition (paired_participants) AND both caches contain can be projected.
projected_participants = [
    participant
    for participant in paired_participants
    if all(participant in cache_row[c.value] for c in CONDITIONS_TO_POOL)
]
dropped = [p for p in paired_participants if p not in projected_participants]
if not projected_participants:
    raise ValueError(
        "No stored participant appears in both caches; there is nothing to project."
    )
if dropped:
    print(f"Not in both caches, dropped: {dropped}")
if N_PAIRS_SUBSET is not None:
    projected_participants = projected_participants[:N_PAIRS_SUBSET]
    print(f"Trimmed to {len(projected_participants)} participant(s)")

# Resolve each (participant, condition) to its store row ONCE; reused by Steps 5 & 10.
store_row = {
    (participant, condition.value): results.row(participant, condition.value)
    for participant in projected_participants
    for condition in CONDITIONS_TO_POOL
}

filtered_by_condition = {}
for condition in CONDITIONS_TO_POOL:
    track = raw_tracks[condition.value]  # (S_c, C, F, T_c)
    projected = np.empty(
        (len(projected_participants), n_components) + track.shape[2:],
        dtype=track.dtype,
    )
    for i, participant in enumerate(projected_participants):
        # (K, C) x (C, F, T) -> (K, F, T): contracts channels only. The filter is the
        # one for THIS (participant, condition) recording.
        projected[i] = np.tensordot(
            spatial_filters[store_row[(participant, condition.value)]],
            track[cache_row[condition.value][participant]],
            axes=([1], [0]),
        )
    filtered_by_condition[condition.value] = projected

# The two outputs, in the segment order the run used.
filtered_placebo = filtered_by_condition[CONDITIONS_TO_POOL[0].value]
filtered_psilocybin = filtered_by_condition[CONDITIONS_TO_POOL[1].value]

print(f"Projected participants : {projected_participants}")
print(
    f"filtered_placebo       : {filtered_placebo.shape}  "
    f"(participants x components x freqs x times)"
)
print(f"filtered_psilocybin    : {filtered_psilocybin.shape}")
print(
    f"Value range            : placebo [{filtered_placebo.min():.3g}, "
    f"{filtered_placebo.max():.3g}], psilocybin "
    f"[{filtered_psilocybin.min():.3g}, {filtered_psilocybin.max():.3g}]"
)
print("Units                  : wavelet power, un-z-scored")

---
## Step 5 — Extract the TF maps with the binary ASSR-electrode filter

A second spatial filter, used exactly where the IVA ones are used. It comes from the
checked-in include-list in
[`config/assr_electrodes/<coordinate system>.csv`](../../config/assr_electrodes) — the
standard fronto-central selection the 40 Hz steady-state response is read from — turned
into a `(channels,)` 0/1 vector by
[`assr_electrode_mask`](../../src/io/loading.py), aligned to the stored run's channel
axis.

It is the same *kind* of operator as an IVA filter: a weighting over channels that
contracts the channel axis and leaves frequency and time untouched. The differences are
what make it worth having alongside:

- **Fixed, not learned.** The weights come from the paradigm and the montage, so they
  are identical for every participant and every condition — no per-participant
  estimation, nothing to sign-align.
- **One variant.** Where IVA gives `K` components, this gives a single map, so the
  output is one `(frequencies, times)` TF map per participant per condition rather than
  `K` of them.

Rows are built from the same participant list and the same cache bookkeeping as Step 4,
so `assr_filtered_placebo[i]` and `filtered_placebo[i]` are the same participant's
Placebo track under the two filters, directly comparable.

**Two references, computed here.** Alongside the mask on the whole raw data (`ASSR-mask (full)`), this step also builds `ASSR-mask (PCA)`: the *same* fronto-central electrode average applied to the raw wavelet data after projecting it onto each participant's per-recording PCA subspace — the exact subspace the IVA components were estimated in. That projector is recovered from the stored arrays as `channel_patterns[p].T @ spatial_filters[p]`, which equals `Pᵀ P` (the whitening in the filter cancels against the pattern), and is checked to be idempotent and symmetric. `ASSR-mask (PCA)` is the like-for-like reference for the components — it can only see what the PCA kept — while `ASSR-mask (full)` shows what that reduction discarded. Being a projected quantity, the PCA row is **signed**.


**Per condition in this variant.** Because the topography is estimated per recording, the PCA subspace — and therefore the `ASSR-mask (PCA)` operator — differs between a participant's Placebo and Psilocybin recordings; `assr_filter_pca_by_condition` holds one per condition. The `ASSR-mask (full)` operator is the fixed electrode average, identical for both.


In [ ]:
# The 0/1 channel selection, aligned to the stored run's channel axis.
assr_mask = assr_electrode_mask(
    list(results.channel_names), COORDINATE_SYSTEM, strict=ASSR_MASK_STRICT
)
assr_channels = [name for name, keep in zip(results.channel_names, assr_mask) if keep]

# One filter row: the K = 1 analogue of spatial_filters, so the projection below is the
# same contraction Step 4 does.
assr_filter = assr_mask.astype(float)
if ASSR_MASK_NORMALIZE:
    assr_filter = assr_filter / assr_filter.sum()

print(
    f"ASSR electrodes : {int(assr_mask.sum())} of {assr_mask.size} channel(s) "
    f"({'mean' if ASSR_MASK_NORMALIZE else 'sum'} over them)"
)
print(f"                  {assr_channels}")

# ── Reference 2: the same mask, but on the PCA-reduced data ───────────
# The IVA components live in each participant's per-recording PCA subspace (n_pca
# directions of the channel space); the mask on the FULL data does not. To read the mask
# the components' own way, the raw wavelet data is first projected onto that subspace.
# The orthogonal project-and-reconstruct operator is recoverable EXACTLY from arrays
# already in hand: with orthonormal-row PCA loadings P_p and filter U_p = W_p P_p, the
# stored pattern is A_p = pinv(U_p) = P_p^T W_p^{-1}, so
#     A_p @ U_p = P_p^T P_p  ==  channel_patterns[p].T @ spatial_filters[p]
# — the whitening W_p cancels and what remains is the PCA subspace projector itself, the
# SAME PCA the preprocessing IVA step used. Checked, not trusted: an orthogonal
# projector must be idempotent and symmetric.
pca_projectors = np.stack(
    [channel_patterns[s].T @ spatial_filters[s] for s in range(n_subjects)]
)  # (S, C, C) per recording, each == P_s^T P_s
proj_idem = max(float(np.abs(pj @ pj - pj).max()) for pj in pca_projectors)
proj_sym = max(float(np.abs(pj - pj.T).max()) for pj in pca_projectors)
print(f"\nPCA subspace projector : {pca_projectors.shape}  (P^T P per recording)")
print(f"  idempotent |PP-P|     : {proj_idem:.2e}")
print(f"  symmetric  |P-P^T|    : {proj_sym:.2e}")
if max(proj_idem, proj_sym) > 1e-5:
    raise ValueError(
        f"The recovered PCA projector is not an orthogonal projector (idempotency "
        f"{proj_idem:.2e}, symmetry {proj_sym:.2e}); the stored patterns and filters "
        "are inconsistent."
    )

# The mask carried into each recording's PCA subspace: (mask @ P_s^T P_s). It is SIGNED
# (projecting a non-negative electrode average onto a subspace can turn it negative, so
# this row behaves like an IC under Step 8) AND per condition, because the subspace is
# estimated per recording — a participant's Placebo and Psilocybin recordings differ.
assr_filter_pca_by_condition = {
    condition.value: np.stack(
        [
            assr_filter @ pca_projectors[store_row[(p, condition.value)]]
            for p in projected_participants
        ]
    )
    for condition in CONDITIONS_TO_POOL
}  # each (P_proj, C)

# ── Project every condition's raw data through BOTH references ─────────
mask_by_condition = {label: {} for label in MASK_LABELS}
for condition in CONDITIONS_TO_POOL:
    track = raw_tracks[condition.value]  # (S_c, C, F, T_c)
    for label in MASK_LABELS:
        projected = np.empty(
            (len(projected_participants),) + track.shape[2:], dtype=track.dtype
        )
        for i, participant in enumerate(projected_participants):
            # (C,) x (C, F, T) -> (F, T): contracts channels only, exactly as the IVA
            # filters do. The full mask is one fixed (C,) vector; both PCA masks use
            # that participant's own subspace-projected version, and differ only in
            # WHAT they are applied to — the raw wavelet, or the z-scored one the
            # decomposition itself was fitted on.
            source = track[cache_row[condition.value][participant]]  # (C, F, T)
            if label == BINARY_FILTER_PCA_Z_LABEL:
                # zscore_by_time, per (channel, frequency), done one participant at a
                # time: a whole z-scored copy of the subset is ~1.8 GB and never has to
                # exist, since only this one contraction reads it.
                spread = source.std(axis=-1, keepdims=True)
                source = (source - source.mean(axis=-1, keepdims=True)) / np.where(
                    spread == 0, 1.0, spread
                )
            channel_op = (
                assr_filter
                if label == BINARY_FILTER_LABEL
                else assr_filter_pca_by_condition[condition.value][i]
            )
            projected[i] = np.tensordot(channel_op, source, axes=([0], [0]))
        mask_by_condition[label][condition.value] = projected

# Back-compatible name for the FULL mask (later steps read mask_by_condition directly).
assr_by_condition = mask_by_condition[BINARY_FILTER_LABEL]

# The FULL-mask TF-map sets kept under their old names (later QC refers to them).
assr_filtered_placebo = assr_by_condition[CONDITIONS_TO_POOL[0].value]
assr_filtered_psilocybin = assr_by_condition[CONDITIONS_TO_POOL[1].value]

print(f"\nProjected participants     : {projected_participants}")
for label in MASK_LABELS:
    pl = mask_by_condition[label][CONDITIONS_TO_POOL[0].value]
    ps = mask_by_condition[label][CONDITIONS_TO_POOL[1].value]
    print(
        f"  {label:<17}: {pl.shape}  (participants x freqs x times); "
        f"placebo [{pl.min():.3g}, {pl.max():.3g}], "
        f"psilocybin [{ps.min():.3g}, {ps.max():.3g}]"
    )
print(
    f"Units                      : wavelet power, un-z-scored — EXCEPT\n"
    f"                             '{BINARY_FILTER_PCA_Z_LABEL}', which is the\n"
    "                             z-scored signal the decomposition was fitted\n"
    "                             on, so it is dimensionless and signed."
)
print(
    f"Note: {BINARY_FILTER_PCA_LABEL} is signed (min can be < 0) because the PCA "
    "projection\n      can turn a non-negative electrode average negative."
)


---
## Step 6 — Put both filters on one source axis and take the 40 Hz band

Steps 4 and 5 produced two arrays of different rank — `(P, K, F, T)` for the learned
IVA components and `(P, F, T)` for the fixed binary mask — because one filter has `K`
rows and the other has one. That difference is an accident of how many rows each
operator happens to have, not a difference in kind: both are a weighting over channels
that contracts the channel axis and leaves frequency and time untouched. So they are
concatenated here into a **single source axis**

```
sources: (participants, K + 1, frequencies, times)
         ^                ^
         |                +-- IC 1 .. IC K, then "ASSR-mask"
         +-- the same participant order as Steps 4-5
```

which makes "the same participant's same trial under each filter" a slice rather than
a join. `SOURCE_LABELS` is what distinguishes them afterwards, and the mask row stays
last so `sources[:, :n_components]` is still exactly the learned set.

**The 40 Hz band, two ways.** The frequency axis is then collapsed, once per entry of
`FREQ_SELECTIONS`:

- **`40hz`** — the single nearest bin. The cache grid is 1 Hz spaced from 1 to 50 Hz,
  so this is the exact 40 Hz row, with no neighbouring frequency mixed in.
- **`35-45hz`** — the mean over every bin inside ±`TF_ANCHOR_HALFWIDTH_HZ`. Wider than
  the response, deliberately: it is insensitive to wavelet smearing and to a few Hz of
  stimulator drift, at the cost of admitting some off-frequency power.

Neither is obviously right, they are cheap, and which one a result survives under is
itself informative — so both are carried through to the trials.

**Still un-z-scored.** Nothing here standardises or baselines anything; the values stay
in the cache's own wavelet-power units.


In [ ]:
# ── One source axis for both kinds of spatial filter ──────────
# The learned components first, in component order, then the single fixed mask row.
SOURCE_LABELS = [f"IC {k + 1}" for k in range(n_components)] + list(MASK_LABELS)

sources_by_condition = {
    condition.value: np.concatenate(
        # the learned components first, then one (P, 1, F, T) row per reference mask.
        [filtered_by_condition[condition.value]]  # (P, K, F, T)
        + [
            mask_by_condition[label][condition.value][:, None]
            for label in MASK_LABELS
        ],
        axis=1,
    )
    for condition in CONDITIONS_TO_POOL
}

n_sources = len(SOURCE_LABELS)
for condition in CONDITIONS_TO_POOL:
    stacked = sources_by_condition[condition.value]
    if stacked.shape[1] != n_sources:
        raise ValueError(
            f"{condition.value}: stacked {stacked.shape[1]} source(s) but there are "
            f"{n_sources} labels."
        )


# ── Collapse the frequency axis, once per selection ───────────
def _selection_bins(center: float, halfwidth: float) -> np.ndarray:
    """Bin indices of ``center +/- halfwidth`` on the cache's frequency grid.

    A zero half-width is the single nearest bin. A positive one takes every bin in the
    closed interval, and falls back to the nearest bin if the interval happens to fall
    between two — an empty selection would silently produce a NaN track.
    """
    if halfwidth < 0:
        raise ValueError(f"Half-width must be >= 0; got {halfwidth}.")
    if halfwidth == 0:
        return np.array([int(np.argmin(np.abs(raw_freqs - center)))])
    inside = np.flatnonzero(
        (raw_freqs >= center - halfwidth) & (raw_freqs <= center + halfwidth)
    )
    return inside if inside.size else np.array([int(np.argmin(np.abs(raw_freqs - center)))])


band_tracks: dict[str, dict[str, np.ndarray]] = {}  # selection -> condition -> (P,S,T)
selection_bins: dict[str, np.ndarray] = {}

for name, center, halfwidth in FREQ_SELECTIONS:
    bins = _selection_bins(center, halfwidth)
    selection_bins[name] = bins
    band_tracks[name] = {
        # Mean over the selected bins; a single-bin selection is that bin verbatim.
        condition.value: sources_by_condition[condition.value][:, :, bins, :].mean(
            axis=2
        )
        for condition in CONDITIONS_TO_POOL
    }

print(f"Sources        : {n_sources}  {SOURCE_LABELS}")
print(
    f"Stacked        : "
    f"{ {c.value: sources_by_condition[c.value].shape for c in CONDITIONS_TO_POOL} }"
)
print(f"Frequency grid : {raw_freqs[0]:.1f}-{raw_freqs[-1]:.1f} Hz ({raw_freqs.size} bins)")
print("\n40 Hz band tracks (participants x sources x times), un-z-scored:")
for name, _center, _halfwidth in FREQ_SELECTIONS:
    bins = selection_bins[name]
    shapes = {c.value: band_tracks[name][c.value].shape for c in CONDITIONS_TO_POOL}
    print(
        f"  {name:<9}: bins {bins.tolist()} = {raw_freqs[bins].tolist()} Hz -> {shapes}"
    )

---
## Step 7 — Cut the 40 Hz tracks into stimulus-locked trials

The step that turns a continuous 40 Hz time course into the unit a trial-level
analysis works with: one fixed window per stimulus onset, **kept separately** rather
than averaged. `iva_quality.epoch_average` already exists for the averaging case; this
is deliberately the other one.

**Where the onsets come from.** Not from the stored decomposition — Step 1 reported
`Onsets stored for: none`, because that run predates onsets being written into the
component store. They come from
`data/processed/<experiment>/concatenated/<Condition>_ASSR.stimulus_onsets.npy`, the
sidecar of the concatenated array. That array is the one the wavelet transform was run
on, so its sample indices address exactly the time axis the projected tracks are on —
no shifting, no rescaling.

**The window comes from the paradigm, not from the recording.**
[`AssrEpoch`](../../src/definitions/constants.py) sets a `PRE_ONSET_S` baseline and a
`POST_ONSET_S` span (stimulus plus an equally long tail), and the recording only
*caps* the latter: `iva_quality.onset_window` shortens `post` by the shortest observed
inter-onset gap so an epoch can never reach the next stimulus. Deriving the length
from the gap alone would make the window a property of whatever jitter this recording
happened to have. The two conditions then share the **shorter** `post` of the two, so
a Placebo trial and a Psilocybin trial are the same number of samples and can be
contrasted sample for sample.

**The 12 s cache is the binding constraint here.** The full recording is 188 s and
carries 148 onsets per condition; the subset cache these tracks came from is 3000
samples (12 s), so only the onsets whose whole window fits inside it survive. The cell
prints how many that is against how many exist, because it is the single most
important caveat on everything downstream — extending it means re-projecting from a
longer cache, not re-cutting these arrays.

**No baseline subtraction.** The pre-onset samples are kept as they are, so a baseline
correction remains a choice the analysis can make (or not) rather than one already
baked in.


In [ ]:
# ── The onsets, from the concatenated array's sidecar ─────────
sidecar_onsets = {}
for condition in CONDITIONS_TO_POOL:
    onsets_path = CONCATENATED_DIR / (
        f"{condition.value}_{MUSIC_TYPE.value}{ProjectPaths.STIMULUS_ONSETS_SUFFIX}"
    )
    if not onsets_path.exists():
        raise FileNotFoundError(
            f"No stimulus onsets for {condition.value} at {onsets_path}. They are "
            "written next to the concatenated array by the stimulus alignment; "
            "without them there are no trials to cut."
        )
    sidecar_onsets[condition.value] = np.load(onsets_path).astype(int)

# ── One epoch window, shared by both conditions ───────────────
# Each condition proposes its own (pre, post) from its own onsets; the shared window is
# the shorter post, so the two conditions' trials are the same length and comparable
# sample for sample.
geometry = {}
for condition in CONDITIONS_TO_POOL:
    track_times = raw_tracks[condition.value].shape[-1]
    onsets = sidecar_onsets[condition.value]
    inside = onsets[(onsets >= 0) & (onsets < track_times)]
    if inside.size == 0:
        raise ValueError(
            f"{condition.value}: none of the {onsets.size} onset(s) falls inside the "
            f"{track_times}-sample cached track."
        )
    pre, post = iva_quality.onset_window(inside, track_times, sfreq)
    geometry[condition.value] = (inside, pre, post, track_times)

epoch_pre = {pre for _o, pre, _post, _n in geometry.values()}
if len(epoch_pre) > 1:
    raise ValueError(
        f"Conditions disagree on the pre-onset baseline ({sorted(epoch_pre)} samples); "
        "they cannot share an epoch window."
    )
EPOCH_PRE = epoch_pre.pop()
EPOCH_POST = min(post for _o, _pre, post, _n in geometry.values())
epoch_times = np.arange(-EPOCH_PRE, EPOCH_POST) / sfreq
stimulus_mask = AssrEpoch.stimulus_mask(epoch_times)  # the driven interval


def _cut_trials(array: np.ndarray, onsets: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """One window per onset from the LAST axis, every trial kept.

    The per-trial counterpart of ``iva_quality.epoch_average``, which collapses the
    same windows to their mean. Windows overhanging either end are dropped, and the
    onsets that survived come back with the data so a trial index is never guessed.
    Returns ``(..., n_trials, EPOCH_PRE + EPOCH_POST)`` — the trial axis sits just
    before time.
    """
    n_times = array.shape[-1]
    kept = np.asarray(
        [
            int(onset)
            for onset in onsets
            if int(onset) - EPOCH_PRE >= 0 and int(onset) + EPOCH_POST <= n_times
        ],
        dtype=int,
    )
    if kept.size == 0:
        raise ValueError(
            f"No {EPOCH_PRE + EPOCH_POST}-sample window fits inside the "
            f"{n_times}-sample track."
        )
    windows = [array[..., o - EPOCH_PRE : o + EPOCH_POST] for o in kept]
    return np.stack(windows, axis=-2), kept


# ── Cut every selection, for every condition ──────────────────
trials: dict[str, dict[str, np.ndarray]] = {}  # selection -> condition -> (P,S,N,W)
trial_onsets: dict[str, np.ndarray] = {}  # condition -> (N,), the kept onsets

for name, _center, _halfwidth in FREQ_SELECTIONS:
    trials[name] = {}
    for condition in CONDITIONS_TO_POOL:
        cut, kept = _cut_trials(
            band_tracks[name][condition.value], geometry[condition.value][0]
        )
        trials[name][condition.value] = cut
        trial_onsets[condition.value] = kept  # identical across selections

print(
    f"Epoch window : {EPOCH_PRE} pre + {EPOCH_POST} post = "
    f"{EPOCH_PRE + EPOCH_POST} samples = "
    f"[{epoch_times[0]:.3f}, {epoch_times[-1]:.3f}] s @ {sfreq} Hz"
)
print(
    f"               stimulus 0-{AssrEpoch.STIMULUS_DURATION_S:.2f} s = "
    f"{int(stimulus_mask.sum())} sample(s) of the window"
)
print("\nOnsets available vs usable, per condition:")
for condition in CONDITIONS_TO_POOL:
    name = condition.value
    inside, _pre, post, track_times = geometry[name]
    total = sidecar_onsets[name].size
    kept = trial_onsets[name]
    print(
        f"  {name:<12}: {total} in the recording, {inside.size} inside the cached "
        f"{track_times} samples ({track_times / sfreq:.1f} s), {kept.size} whole "
        f"window(s) kept"
    )
    if post < AssrEpoch.post_onset_samples(sfreq):
        print(
            f"                post trimmed to {post} samples "
            f"({post / sfreq:.3f} s) by the shortest inter-onset gap"
        )
if any(trial_onsets[c.value].size < MIN_ONSETS_FOR_EPOCH_AVERAGE for c in CONDITIONS_TO_POOL):
    print(
        f"\n  WARNING: fewer than MIN_ONSETS_FOR_EPOCH_AVERAGE "
        f"({MIN_ONSETS_FOR_EPOCH_AVERAGE}) trials in at least one condition."
    )

print("\nTrials (participants x sources x trials x samples), un-z-scored, no baseline:")
for name, _center, _halfwidth in FREQ_SELECTIONS:
    shapes = {c.value: trials[name][c.value].shape for c in CONDITIONS_TO_POOL}
    print(f"  {name:<9}: {shapes}")

---
## Step 8 — Normalise every trial by its own pre-stimulus interval

Each trial is referenced to its **own** `times < 0` window (25 samples, −0.100 to
−0.004 s) rather than to a condition- or participant-level average. That is what makes
this worth doing: the level of 40 Hz power drifts across the recording and differs by
participant, and a per-trial baseline removes both without any group statistic entering
the correction.

**Two forms, because one form does not fit both kinds of source.** The diagnostic that
decides this is in the cell below, and it is not a detail:

| source | per-trial baselines that are negative | worst \|x / baseline\| |
|---|---|---|
| `IC 1`–`IC 5` | 20–56 of 108 per condition | 27 … 3790 |
| `ASSR-mask` | 0 of 108 | 8.5 / 23.4 |

The mask is a non-negative average over electrodes, so dividing by its baseline is
exactly the conventional relative-power change. An IVA source is a **signed**
combination of channels: its power runs negative on roughly half the trials, and a few
baselines sit near zero. Dividing by those does not give a noisy answer, it gives a
meaningless one — the sign flips wherever the baseline is negative, and the magnitude
explodes wherever it is small. So:

- **`z`** — `(x − baseline) / baseline SD`, each trial in units of its own pre-stimulus
  variability. Defined for **every** trial of **every** source regardless of sign,
  dimensionless, and comparable across participants and sources. **This is the
  canonical form: use it for the ICs and for `ASSR-mask` alike.**
- **`rel`** — `x / baseline − 1`, the relative change, as a human-readable percentage.
  Written **only where the baseline is positive**; every other trial is `NaN`,
  deliberately, so a downstream mean cannot quietly average a sign-flipped value. That
  makes it complete for `ASSR-mask` and a *biased subset* on the IC rows, where roughly
  half the baselines are negative. Treat it as a readout for the mask, never as the
  input to a comparison that spans both kinds of source.

**Why one form for everything, rather than the best form for each row.** The point of
this extraction is to put the learned components and the fixed mask side by side and
ask which recovers the response better. Normalising them differently would confound
exactly that comparison: a difference between the two arms could then come from the
transform rather than from the spatial filter, and the two arms' effect sizes would not
even be in the same units. `z` is the form both rows can carry, so `z` is the one the
comparison runs on. `rel` stays in the file because a percentage change is easier to
report than a z-score — not because the analysis should switch between them.

Both are stored next to the raw power, so nothing is thrown away and the choice stays
reversible.

**This is not cosmetic — it changes the result.** Averaged as raw power, Placebo showed
a modest onset-locked rise while Psilocybin appeared to decline monotonically across the
whole epoch, which read as a slow non-stationarity swamping any response. That reading
was an artefact of the averaging: the trials differ in *level*, and with 9 trials the
group mean of raw power is dominated by those level differences rather than by the
stimulus-locked change inside each trial. Referencing every trial to its own baseline
first removes them, and **both** conditions then show a clear driven response peaking
inside the 0–0.5 s stimulus interval (Step 9). Treat any conclusion drawn from the
un-normalised group mean as superseded.

**What it still does not fix.** A drift *within* the 1.1 s epoch survives, because the
baseline only sets each trial's starting level. And normalising cannot recover
statistical power: 9 trials per participant is what the 12 s subset cache allows, and
the post-stimulus interval does not return to zero here, so the epochs are not fully
independent of one another.

**The PCA reference is a signed row.** The table above characterises `IC 1`–`IC K` and `ASSR-mask (full)`; `ASSR-mask (PCA)` behaves like an IC here, not like the full mask — projecting the non-negative electrode average onto the PCA subspace can make it negative, so its per-trial baseline goes negative on some trials and `rel` is `NaN` there. Use `z` for it, as for the ICs.


In [ ]:
# The pre-stimulus interval each trial is referenced to.
baseline_mask = epoch_times < 0.0
print(
    f"Pre-stimulus interval: {int(baseline_mask.sum())} samples, "
    f"{epoch_times[0]:.3f} to {epoch_times[baseline_mask][-1]:.3f} s"
)

# ── Why two forms: how often is a per-trial baseline unusable as a divisor? ──
print("\nPer-trial baselines, by source (a ratio needs a POSITIVE baseline):")
print(
    f"  {'source':<10} {'condition':<11} {'negative':>10} {'near zero':>11} {'max |ratio|':>12}"
)
for s, source in enumerate(SOURCE_LABELS):
    for condition in CONDITIONS_TO_POOL:
        arr = trials[FREQ_SELECTIONS[0][0]][condition.value][:, s]  # (P, N, W)
        base = arr[..., baseline_mask].mean(axis=-1)
        # "Near zero" relative to the trial's own typical magnitude, which is what
        # decides whether the division explodes.
        near_zero = np.abs(base) < 0.1 * np.abs(arr).mean(axis=-1)
        print(
            f"  {source:<10} {condition.value:<11} "
            f"{f'{int((base < 0).sum())}/{base.size}':>10} "
            f"{f'{int(near_zero.sum())}/{base.size}':>11} "
            f"{np.abs(arr / base[..., None]).max():>12.3g}"
        )


def _normalise(array: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Per-trial baseline normalisation of a ``(P, S, N, W)`` array.

    Returns ``(rel, z, baseline_positive)``:

    * ``rel`` — ``x / baseline - 1``, NaN wherever the baseline is not positive. The
      NaN is the point: on a signed IVA source a negative baseline silently INVERTS
      the ratio, so those trials must not survive into a mean as numbers.
    * ``z`` — ``(x - baseline) / baseline SD``, defined for every trial whatever its
      sign, in units of the source's pre-stimulus variability (pooled across trials).
    * ``baseline_positive`` — ``(P, S, N)`` mask of the trials ``rel`` is valid for.
    """
    base = array[..., baseline_mask].mean(axis=-1, keepdims=True)
    # The MEAN is per trial (drift is local and has to be removed locally); the SD is
    # pooled ACROSS trials. At 25 baseline samples a mean is a sound estimator but an SD
    # is not — it varies ~4x more than sampling theory allows and is heavy-tailed, so a
    # trial that draws a quiet baseline gets divided by a spuriously small number, which
    # inflates that trial's WHOLE epoch. Pooling estimates the same quantity from
    # trials x baseline samples instead of 25. Same convention as the ROI reference row,
    # so every source on the axis is in the same units.
    per_trial = array[..., baseline_mask].std(axis=-1, ddof=1, keepdims=True)
    spread = np.sqrt((per_trial**2).mean(axis=-2, keepdims=True))

    positive = base > 0
    rel = np.where(positive, array / np.where(positive, base, 1.0) - 1.0, np.nan)

    # A zero-variance baseline would divide by zero; it never occurs on real wavelet
    # power, but guard rather than emit a silent inf.
    usable = spread > 0
    z = np.where(usable, (array - base) / np.where(usable, spread, 1.0), np.nan)
    return rel, z, positive[..., 0]


trials_rel: dict[str, dict[str, np.ndarray]] = {}
trials_z: dict[str, dict[str, np.ndarray]] = {}
baseline_positive: dict[str, dict[str, np.ndarray]] = {}

for name, _center, _halfwidth in FREQ_SELECTIONS:
    trials_rel[name], trials_z[name], baseline_positive[name] = {}, {}, {}
    for condition in CONDITIONS_TO_POOL:
        rel, z, positive = _normalise(trials[name][condition.value])
        trials_rel[name][condition.value] = rel
        trials_z[name][condition.value] = z
        baseline_positive[name][condition.value] = positive

print("\nNormalised arrays, same (participants x sources x trials x samples) layout:")
for name, _center, _halfwidth in FREQ_SELECTIONS:
    for condition in CONDITIONS_TO_POOL:
        rel = trials_rel[name][condition.value]
        valid = baseline_positive[name][condition.value]
        print(
            f"  {name:<9} {condition.value:<11} rel {rel.shape}  "
            f"{int(valid.sum())}/{valid.size} trial(s) with a positive baseline "
            f"({100 * valid.mean():.0f}%), the rest NaN in `rel`; `z` is complete"
        )

# ── Rebuild the binary ROI row with equal weight per electrode ──
# Step 5 averaged the raw power over the ROI and this cell normalised the average, which
# silently weights each electrode by its own power level. This replaces that one column
# with the equal-weight construction (see ROI_CHANNELWISE), leaving every other row
# untouched. Both `prestim` and `stored_prestim` read their reference from `trials_z`,
# so patching it here is enough for both.
if ROI_CHANNELWISE:
    roi_index = np.flatnonzero(assr_mask)
    full_col = SOURCE_LABELS.index(BINARY_FILTER_LABEL)
    print(
        f"\nEqual-weight ROI row over {roi_index.size} electrode(s) "
        f"(replacing '{BINARY_FILTER_LABEL}'):"
    )
    for name, _center, _halfwidth in FREQ_SELECTIONS:
        bins = selection_bins[name]
        # Two passes. The first builds each condition's row on its OWN scale; the second
        # puts them all on one shared constant. Leaving them apart would rescale a
        # participant's two conditions differently — by up to 63% on this cohort — and
        # that lands straight in the paired difference the contrast tests. The rescale
        # is exact rather than a re-fit: snr / shared == (snr / own) * (own / shared).
        per_condition, scales = {}, {}
        for condition in CONDITIONS_TO_POOL:
            cond = condition.value
            rows = [cache_row[cond][p] for p in projected_participants]
            roi_band = raw_tracks[cond][rows][:, roi_index][:, :, bins].mean(axis=2)
            roi_cut, _kept = _cut_trials(roi_band, geometry[cond][0])  # (P, R, N, W)
            per_condition[cond], scales[cond] = at.roi_channelwise_snr(
                roi_cut, baseline_mask
            )
        shared_scale = float(np.median([d["roi_baseline_sd"] for d in scales.values()]))
        for condition in CONDITIONS_TO_POOL:
            cond = condition.value
            roi_diag = scales[cond]
            roi_snr = per_condition[cond] * (roi_diag["roi_baseline_sd"] / shared_scale)
            trials_z[name][cond][:, full_col] = roi_snr
            # `rel` has no equal-weight analogue — a ratio of z-scores is meaningless —
            # so mark that column unusable rather than leave a stale power-weighted one.
            trials_rel[name][cond][:, full_col] = np.nan
            baseline_positive[name][cond][:, full_col] = False
            if name == FREQ_SELECTIONS[0][0]:
                print(
                    f"  {name:<9} {cond:<11} own baseline SD "
                    f"{roi_diag['roi_baseline_sd']:.3f} -> shared {shared_scale:.3f}; "
                    f"per-participant spread "
                    f"{roi_diag['participant_sd_spread']:.2f}x (NOT applied); "
                    f"max |per-channel z| {roi_diag['max_abs_z1']:.0f}"
                )
    print(
        "  One shared constant per selection puts this row in units of its "
        "own baseline SD, so it\n  is comparable with the IC rows Step 12 tests it "
        "against — WITHOUT rescaling any\n  participant or condition relative to "
        "another, which a per-participant divisor\n  would do, and which would corrupt "
        "the paired contrast."
    )

# Per source, so it is obvious which rows `rel` can actually be used on.
print("\nShare of trials with a usable (positive) baseline, per source:")
for s, source in enumerate(SOURCE_LABELS):
    shares = {
        condition.value: baseline_positive[FREQ_SELECTIONS[0][0]][condition.value][
            :, s
        ].mean()
        for condition in CONDITIONS_TO_POOL
    }
    verdict = (
        "rel usable"
        if min(shares.values()) > 0.99
        else "USE z — rel is NaN on much of this row"
    )
    joined = "  ".join(f"{c} {v:5.0%}" for c, v in shares.items())
    print(f"  {source:<10} {joined}   {verdict}")


---
## Step 9 — Write the extracted trials, and check they look like a response

One `.npz` per frequency selection, each **self-contained**: the trial arrays plus
every axis needed to interpret them — participant labels, source labels, the epoch time
base, the kept onsets, the frequency bins that were averaged, and the provenance of the
filters. Nothing downstream should have to re-derive a row's meaning from this
notebook's variable names.

Per-condition arrays are keyed `trials__<Condition>` rather than stacked into one
array, because the two conditions are free to keep a different number of trials and
padding them to a common length would invent data.

```python
with np.load(path, allow_pickle=True) as f:
    raw = f["trials__Placebo"]          # (participants, sources, trials, samples)
    rel = f["trials_rel__Placebo"]      # x / baseline - 1, NaN where baseline <= 0
    z   = f["trials_z__Placebo"]        # (x - baseline) / baseline SD, always defined
    labels = f["source_labels"]         # "IC 1" .. "IC K", "ASSR-mask"
    t      = f["times"]                 # seconds, 0 at onset
```

Each file carries the raw power **and** both per-trial normalisations from Step 8, so the choice of reference stays reversible.

The final output is a **sanity check on the extraction, not the analysis**: for each
source, the group-mean power in the driven interval against the pre-onset baseline,
plus the `ASSR-mask` time course printed sample by sample. What is being checked is
that a stimulus-locked change exists at all and sits in the right place — rising after
onset and falling back around the 0.5 s stimulus offset. If it does not, the epochs are
mis-timed or the rows are mis-indexed, and no downstream contrast is worth running. No
condition difference is tested here.

**Why a relative change and not a ratio.** The `ASSR-mask` row is a *non-negative*
average over electrodes, so a ratio would be fine for it. An IVA source is not: the
filter is a **signed** combination of channels, so a component's "power" can be
negative, and the ratio of two such numbers is uninterpretable — arbitrarily large,
negative, or near-infinite purely from a baseline close to zero. `(stimulus −
baseline) / |baseline|` is well defined for both kinds of row, so both can sit in one
table. Read the mask row first regardless: it is fixed by the montage and the paradigm,
so it is the assumption-free reference the learned rows are judged against.


In [ ]:
# ── Write one self-contained file per frequency selection ─────
# Provenance shared by every file: enough to tell which decomposition these filters
# came from and what was (not) done to the signal they were applied to.
zscore_note = zscore_mode.item() if zscore_mode is not None else "unrecorded"
trial_metadata = {
    "store_path": str(results.path),
    "variant": results.variant.value,
    "store_condition": results.condition.value,
    "music_type": MUSIC_TYPE.value,
    "n_pca": str(N_COMPONENTS_PCA),
    "filter_zscore_mode": zscore_note,
    "signal_zscore": "none — trials are raw wavelet power from the subset cache",
    "baseline_correction": "none",
    "units": "wavelet power",
    "binary_filter_full": "mean" if ASSR_MASK_NORMALIZE else "sum",
    "binary_filter_pca": (
        "full mask projected onto each participant's IVA PCA subspace (P^T P); signed"
    ),
    "cache_n_times": str(raw_tracks[CONDITIONS_TO_POOL[0].value].shape[-1]),
    "sign_alignment": ALIGNMENT_NOTE,
}

trial_paths = []
if SAVE_TRIALS:
    TRIALS_DIR.mkdir(parents=True, exist_ok=True)
for name, _center, _halfwidth in FREQ_SELECTIONS:
    payload = {
        "participants": np.asarray(projected_participants, dtype=object),
        "source_labels": np.asarray(SOURCE_LABELS, dtype=object),
        "conditions": np.asarray([c.value for c in CONDITIONS_TO_POOL], dtype=object),
        "times": epoch_times,
        "sfreq": np.asarray(sfreq),
        "freqs": raw_freqs[selection_bins[name]],
        "selection": np.asarray(name),
        "epoch_pre": np.asarray(EPOCH_PRE),
        "epoch_post": np.asarray(EPOCH_POST),
        "binary_channels": np.asarray(assr_channels, dtype=object),
        "channel_names": np.asarray(list(results.channel_names), dtype=object),
    }
    for condition in CONDITIONS_TO_POOL:
        payload[f"trials__{condition.value}"] = trials[name][condition.value]
        # Both per-trial normalisations from Step 8, so nothing is thrown away.
        payload[f"trials_rel__{condition.value}"] = trials_rel[name][condition.value]
        payload[f"trials_z__{condition.value}"] = trials_z[name][condition.value]
        payload[f"baseline_positive__{condition.value}"] = baseline_positive[name][
            condition.value
        ]
        payload[f"onsets__{condition.value}"] = trial_onsets[condition.value]
    for key, value in trial_metadata.items():
        payload[f"meta__{key}"] = np.asarray(str(value))

    path = TRIALS_DIR / (
        f"assr_trials__{VARIANT.value}__{MUSIC_TYPE.value}__{name}"
        f"__pca{N_COMPONENTS_PCA}.npz"
    )
    if SAVE_TRIALS:
        np.savez_compressed(path, **payload)
        print(f"saved {path}  ({path.stat().st_size / 1e6:.2f} MB)")
    trial_paths.append(path)
if not SAVE_TRIALS:
    print("SAVE_TRIALS is False — nothing written; the arrays live in `trials`.")

# ── QC: does the driven interval stand out from the baseline? ─────────
# Computed on the PER-TRIAL normalised arrays from Step 8, not on raw power. Averaging
# raw power over trials is dominated by between-trial level differences, which is
# exactly what the normalisation removes — a QC table built on it would contradict the
# time course printed below. baseline_mask and stimulus_mask come from Steps 7-8.
records = []
for name, _center, _halfwidth in FREQ_SELECTIONS:
    for condition in CONDITIONS_TO_POOL:
        rel = trials_rel[name][condition.value]  # (P, S, N, W)
        z = trials_z[name][condition.value]
        for s, source in enumerate(SOURCE_LABELS):
            records.append(
                {
                    "selection": name,
                    "condition": condition.value,
                    "source": source,
                    # nanmean: `rel` is NaN wherever the baseline was not positive.
                    "rel": np.nanmean(rel[:, s][..., stimulus_mask]),
                    "z": np.nanmean(z[:, s][..., stimulus_mask]),
                }
            )
summary = pd.DataFrame.from_records(records)
n_trials = trial_onsets[CONDITIONS_TO_POOL[0].value].size
print(
    f"\nStimulus-interval mean of the per-trial normalised trials, "
    f"{len(projected_participants)} participants x {n_trials} trials — QC of the "
    f"extraction, not a test:"
)
display(
    summary.pivot_table(
        index=["selection", "source"], columns="condition", values=["rel", "z"]
    ).round(3)
)
print(
    "  Read the `z` block: it is the one form every source carries, so it is what a\n"
    "  mask-vs-IC comparison must run on. `rel` on an IC row is a nanmean over only\n"
    "  its positive-baseline trials — a biased subset — so it is a readout for\n"
    "  ASSR-mask only, never the input to a comparison spanning both."
)

# The shape over the epoch, now on the PER-TRIAL normalised data. A driven 40 Hz
# response should rise after onset and fall back around the stimulus offset; a
# monotonic drift across the whole window is a slow non-stationarity of the recording
# that per-trial baselining re-references to zero but does not remove.
marks = np.linspace(0, epoch_times.size - 1, 23).astype(int)
selection = FREQ_SELECTIONS[0][0]
for mask_label in MASK_LABELS:
    mask_row = SOURCE_LABELS.index(mask_label)
    print(
        f"\nPer-trial normalised {mask_label} at "
        f"{raw_freqs[selection_bins[selection]].tolist()} Hz, group+trial mean "
        "(0 = its own pre-stimulus level):"
    )
    print("  t (s)      : " + " ".join(f"{epoch_times[i]:6.2f}" for i in marks))
    for condition in CONDITIONS_TO_POOL:
        rel = np.nanmean(
            trials_rel[selection][condition.value][:, mask_row], axis=(0, 1)
        )
        print(
            f"  {condition.value[:6]:<6} rel: "
            + " ".join(f"{rel[i]:6.2f}" for i in marks)
        )
    for condition in CONDITIONS_TO_POOL:
        z = np.nanmean(trials_z[selection][condition.value][:, mask_row], axis=(0, 1))
        print(
            f"  {condition.value[:6]:<6} z  : "
            + " ".join(f"{z[i]:6.2f}" for i in marks)
        )
print(f"  stimulus interval: 0.00-{AssrEpoch.STIMULUS_DURATION_S:.2f} s")

---
## Step 9b — The learned rows, cut straight out of the stored decomposition

Steps 4-8 reached the per-trial 40 Hz time courses the long way round: recover each
recording's spatial filter from the stored patterns, read the cached raw wavelet back,
apply the filter, cut trials, normalise. This step reaches the same quantity the short
way for the **learned rows only** — the stored `tf_maps` *are* the filtered signal, so
the projection has already happened and only the cutting and normalising are left.

In this variant every array is indexed by **recording**, not participant, so a row is
resolved through `store_row[(participant, condition)]`: a participant's Placebo source
and Psilocybin source are two different rows of `tf_maps`, produced by two different
mixing matrices. That is the whole difference from the time-concatenated sibling, where
one row serves both conditions.

**The references do not move.** Both ASSR-mask rows are taken verbatim from Step 8:
the raw wavelet power converted by the fixed electrode mask and referenced to each
trial's own pre-stimulus window, exactly the array the `prestim` variant tests. They are
the fixed quantity a component is judged against, so holding them still is what makes
the `prestim` / `stored_prestim` comparison mean one thing — the component is read a
different way, and nothing else changed.

The two routes to a component are not the same number, and the difference is the point:

- The decomposition was fitted on data **z-scored along time**, so the stored source is
  `U_s @ z_s`, i.e. a combination of channels weighted by `U_s[k, c] / sd(x_sc)`.
  Step 4 applies the same `U_s` to the **un-z-scored** wavelet, which drops that
  per-channel rescaling. Step 4's output is therefore an approximation of the source;
  the stored array is the source itself.
- The stored array spans the **whole 188 s recording**, so every one of the ~148
  stimulus onsets yields a trial. The cached subset the projection route reads is 12 s
  long and yields ~9. Sixteen times the trials, for no extra I/O.

Note that the per-condition-mixing-matrix worry that applies to the `_tracks` sibling —
where a single `U` is shared and only the z-scoring differs by condition — does not
arise here in the same form: this variant estimates a separate decomposition per
recording by construction, so both routes already give the two conditions different
spatial filters.

**Why the per-trial baseline is what makes this usable.** A stored source is signed and
sits at roughly zero over the recording, because that is what z-scoring along time
leaves behind; its absolute level carries no information at all. Referencing each trial
to its own pre-stimulus window converts it into something that does: a response measured
in units of that trial's own pre-stimulus variability, i.e. a signal-to-noise ratio —
the same unit the reference rows are already in, which is what lets the two be drawn on
one axis. The `rel` form (`x / baseline - 1`) is meaningless here, since around half of
the baselines are negative, so this step keeps only the `z` form, exactly as Step 8
recommends for the signed rows.

The one thing to keep in view when reading Step 12: with `STORED_PRESTIM_ONSETS = "all"`
an IC row averages ~148 trials and a reference row ~9, so the IC side of an
IC-minus-reference gap is the less noisy of the two. The trial counts are printed below
for exactly that reason.


In [ ]:
# ── Which store row each (participant, condition) is ─────────
# Unlike the time-concatenated sibling, a participant has TWO rows here — one per
# recording — so the resolver is keyed by the pair and the source is selected inside the
# per-condition loop rather than once.
store_rows_by_condition = {
    condition.value: [
        store_row[(participant, condition.value)]
        for participant in projected_participants
    ]
    for condition in CONDITIONS_TO_POOL
}

# ── Which onsets the stored IC rows are cut at ────────────────
if STORED_PRESTIM_ONSETS == "all":
    stored_onsets = {c.value: sidecar_onsets[c.value] for c in CONDITIONS_TO_POOL}
elif STORED_PRESTIM_ONSETS == "subset":
    # The subset onsets index the same origin as the full ones — the cache is the FIRST
    # N samples of the aligned time axis — so they address the stored track unchanged.
    stored_onsets = {c.value: trial_onsets[c.value] for c in CONDITIONS_TO_POOL}
else:
    raise ValueError(
        f"STORED_PRESTIM_ONSETS must be 'all' or 'subset'; got "
        f"{STORED_PRESTIM_ONSETS!r}."
    )


def _stored_bins(center: float, halfwidth: float) -> np.ndarray:
    """``_selection_bins`` on the STORE's frequency grid rather than the cache's.

    The two grids are usually the same, but a band-restricted run or a differently
    trimmed cache would make them differ, and a selection has to be resolved against the
    axis of the array it is taken from.
    """
    if halfwidth == 0:
        return np.array([int(np.argmin(np.abs(freqs - center)))])
    inside = np.flatnonzero(
        (freqs >= center - halfwidth) & (freqs <= center + halfwidth)
    )
    return inside if inside.size else np.array([int(np.argmin(np.abs(freqs - center)))])


# ── Cut and normalise the learned rows; carry the references over ──
# Per SOURCE rather than one (P, S, N, W) block: the ICs are cut on the whole recording
# and the references on the 12 s subset, so the trial axes have different lengths and
# cannot share an array.
stored_trials_z: dict[str, dict[str, dict[str, np.ndarray]]] = {}
stored_bins: dict[str, np.ndarray] = {}
stored_trial_counts: dict[str, dict[str, int]] = {}
stored_negative_baselines: dict[str, tuple[int, int]] = {}

for name, center, halfwidth in FREQ_SELECTIONS:
    bins = _stored_bins(center, halfwidth)
    stored_bins[name] = bins
    stored_trials_z[name] = {}
    for condition in CONDITIONS_TO_POOL:
        cond = condition.value
        per_source: dict[str, np.ndarray] = {}

        # Collapse frequency BEFORE selecting rows: the full (S, K, F, T) block is
        # ~1 GB and the selected band is a few MB.
        src = tf_maps[:, :, bins, :].mean(axis=2)[
            store_rows_by_condition[cond]
        ]  # (P, K, T)
        cut, kept = _cut_trials(src, stored_onsets[cond])  # (P, K, N, W)
        _rel, z, _positive = _normalise(cut)
        for k in range(n_components):
            per_source[f"IC {k + 1}"] = z[:, k]

        # ── The references, verbatim from Step 8 ──────────────
        # The SAME array the "prestim" variant tests: raw wavelet power converted by the
        # mask, each trial referenced to its own pre-stimulus window. Not recomputed
        # here — sliced out of trials_z — so the two variants cannot drift apart.
        for label in MASK_LABELS:
            per_source[label] = trials_z[name][cond][:, SOURCE_LABELS.index(label)]

        stored_trials_z[name][cond] = per_source
        if name == FREQ_SELECTIONS[0][0]:
            base = cut[..., baseline_mask].mean(axis=-1)
            stored_negative_baselines[cond] = (int((base < 0).sum()), int(base.size))
            stored_trial_counts[cond] = {
                label: arr.shape[1] for label, arr in per_source.items()
            }

missing = [
    s
    for s in SOURCE_LABELS
    if s not in stored_trials_z[FREQ_SELECTIONS[0][0]][CONDITIONS_TO_POOL[0].value]
]
if missing:
    raise ValueError(f"Step 9b produced no row for {missing}.")

print(f"Onset scope  : {STORED_PRESTIM_ONSETS}  (IC rows only)")
print(f"References   : {MASK_LABELS} — taken from Step 8 unchanged, raw wavelet")
print(f"Epoch window : reused from Step 7, {EPOCH_PRE} pre + {EPOCH_POST} post")
print("\nTrials per row (participants x trials x samples), pre-stimulus normalised:")
for condition in CONDITIONS_TO_POOL:
    cond = condition.value
    negative, total = stored_negative_baselines[cond]
    print(f"  {cond}  (store rows {store_rows_by_condition[cond]}):")
    for label in SOURCE_LABELS:
        arr = stored_trials_z[FREQ_SELECTIONS[0][0]][cond][label]
        origin = "Step 8 (raw)" if label in MASK_LABELS else "store"
        print(f"    {label:<17} {str(arr.shape):<16} {origin}")
    print(
        f"    negative baselines on the stored rows: {negative}/{total} "
        f"({100 * negative / total:.0f}%) — signed sources, so `z` is the only usable "
        "form, never `rel`"
    )


---
## Step 10 — Participant-level tests: condition contrast, and mask vs IC

The reductions, then the statistics. Each source's stimulus-window response is collapsed
to ONE number per (participant, condition), and the **participants are the unit of every
test** — never the trials: a participant's trials are correlated, so treating them as
independent is anticonservative. IVA couples component *k* across all recordings (that is
what IVA buys over per-dataset ICA), so IC *k* under Placebo and under Psilocybin is the
same component and a paired contrast on it is meaningful.

**Polarity is re-anchored to the ASSR electrodes, per recording.** Because the topography
is estimated per recording in this variant, a component's sign is resolved for each
(participant, condition) from that recording's own pattern:

```
flip[p, c, k] = sign( mean forward-pattern weight of IC k over the ASSR electrodes )
```

so a higher value means more 40 Hz power over that area for every row. This reads only the
fixed spatial reference (the pattern's ASSR-electrode projection), never the tested
response, so it cannot manufacture a condition difference — an anchor taken from the tested
quantity would, and inflates the one-sided false-positive rate to ~0.68 under a simulated
null. The flip is applied to both conditions' values symmetrically.

**Exact Wilcoxon signed-rank**, paired over participants, enumerating all `2**P` sign
assignments over the ranks — the p is exact and free of any distributional assumption, with
a hard floor of `2/2**P` (two-sided).

**Two families, both uncorrected.**

- **4b — Placebo vs Psilocybin**, one test per source (one-sided `greater`, the prior that
  psilocybin lowers the 40 Hz response, made interpretable by the polarity anchor).
- **4c — does an IC separate the conditions better than a reference?** The interaction
  `(IC placebo − IC psilocybin) − (mask placebo − mask psilocybin)`, one test per IC against
  **each** reference (`ASSR-mask (full)` and `ASSR-mask (PCA)`), two-sided.

**Two signal variants**, run side by side (`TEST_VARIANTS`): `zscored` reads the stored
per-recording z-scored sources for the ICs (full recording, all onsets) and the mask on the
z-scored 12 s subset; `prestim` reads the raw subset projected and per-trial baselined.
`TEST_FREQ_WINDOW` and `TEST_STIMULUS_INTERVAL` set the frequency band and time window the
test reduces over.


In [ ]:
# ── Resolve the test's frequency bins, stimulus window and variants ──
if TEST_SELECTION not in trials_z:
    raise KeyError(
        f"TEST_SELECTION {TEST_SELECTION!r} is not among the extracted selections "
        f"{list(trials_z)}. Add it to FREQ_SELECTIONS, or set TEST_FREQ_WINDOW."
    )
_KNOWN_VARIANTS = ("zscored", "prestim", "stored_prestim")
for _variant in TEST_VARIANTS:
    if _variant not in _KNOWN_VARIANTS:
        raise ValueError(
            f"TEST_VARIANTS entries must be one of {_KNOWN_VARIANTS}; got {_variant!r}."
        )
if "stored_prestim" in TEST_VARIANTS and TEST_SELECTION not in stored_trials_z:
    raise KeyError(
        f"TEST_SELECTION {TEST_SELECTION!r} was not extracted by Step 9b "
        f"(it has {list(stored_trials_z)}). Re-run Step 9b after changing "
        "FREQ_SELECTIONS or TEST_FREQ_WINDOW."
    )

test_freq_bins = selection_bins[TEST_SELECTION]

# The stimulus window the test reads: a custom interval, or the paradigm default.
if TEST_STIMULUS_INTERVAL is None:
    test_stim_mask = stimulus_mask
    test_window_desc = f"0-{AssrEpoch.STIMULUS_DURATION_S:.3f} s (paradigm default)"
else:
    _lo, _hi = TEST_STIMULUS_INTERVAL
    test_stim_mask = (epoch_times >= _lo) & (epoch_times <= _hi)
    if not test_stim_mask.any():
        raise ValueError(
            f"TEST_STIMULUS_INTERVAL {TEST_STIMULUS_INTERVAL} selects no epoch sample; "
            f"the epoch spans [{epoch_times[0]:.3f}, {epoch_times[-1]:.3f}] s."
        )
    test_window_desc = f"{_lo:.3f}-{_hi:.3f} s (custom)"
test_rest_mask = ~test_stim_mask

n_test_participants = len(projected_participants)
n_test_trials = trials_z[TEST_SELECTION][CONDITIONS_TO_POOL[0].value].shape[2]
print(f"Test frequency : {TEST_SELECTION}  ({raw_freqs[test_freq_bins].tolist()} Hz)")
print(f"Test window    : {test_window_desc}, {int(test_stim_mask.sum())} sample(s)")
print(f"Response       : {RESPONSE_MEASURE}")
print(f"Variants       : {TEST_VARIANTS}")


def _reduce_window(windowed: np.ndarray) -> np.ndarray:
    """Collapse the epoch-time axis of a ``(..., W)`` array to one number per trial."""
    if RESPONSE_MEASURE == "stimulus":
        return windowed[..., test_stim_mask].mean(axis=-1)
    if RESPONSE_MEASURE == "stimulus_minus_rest":
        return windowed[..., test_stim_mask].mean(axis=-1) - windowed[
            ..., test_rest_mask
        ].mean(axis=-1)
    raise ValueError(
        f"RESPONSE_MEASURE must be 'stimulus' or 'stimulus_minus_rest'; got "
        f"{RESPONSE_MEASURE!r}."
    )


EPOCH_LEN = EPOCH_PRE + EPOCH_POST


def _cut_windows(band_course: np.ndarray, onsets: np.ndarray):
    """Stack one ``(W,)`` window per fitting onset from a ``(T,)`` band course."""
    n_times = band_course.shape[-1]
    kept = [
        int(o)
        for o in onsets
        if int(o) - EPOCH_PRE >= 0 and int(o) + EPOCH_POST <= n_times
    ]
    if not kept:
        return None
    return np.stack([band_course[o - EPOCH_PRE : o + EPOCH_POST] for o in kept])


# ── Build per-participant value (P, S) and epoch course (P, S, W) per variant ──
# The two variants differ ONLY in how each source's signal is put into comparable units;
# both then reduce over the same frequency bins, the same stimulus window and the median
# over trials. "prestim" uses the per-trial baseline-normalised subset trials from
# Step 8. "zscored" uses the STORED z-scored IVA sources (full recording, all onsets) for
# the IC rows and the mask applied to the z-scored 12 s subset for the reference rows —
# so its IC and mask rows carry different trial counts and cannot share one array, which
# is why the value is assembled column by column.
value_by_variant: dict[str, dict[str, np.ndarray]] = {}
course_by_variant: dict[str, dict[str, np.ndarray]] = {}

c0 = CONDITIONS_TO_POOL[0].value
c1 = CONDITIONS_TO_POOL[1].value

if "prestim" in TEST_VARIANTS:
    value_by_variant["prestim"] = {}
    course_by_variant["prestim"] = {}
    for condition in CONDITIONS_TO_POOL:
        z = trials_z[TEST_SELECTION][condition.value]  # (P, S, N, W)
        value_by_variant["prestim"][condition.value] = np.median(
            _reduce_window(z), axis=2
        )  # (P, S)
        course_by_variant["prestim"][condition.value] = np.median(z, axis=2)  # (P,S,W)

if "zscored" in TEST_VARIANTS:
    value_z = {
        condition.value: np.full((n_test_participants, n_sources), np.nan)
        for condition in CONDITIONS_TO_POOL
    }
    course_z = {
        condition.value: np.full((n_test_participants, n_sources, EPOCH_LEN), np.nan)
        for condition in CONDITIONS_TO_POOL
    }
    # z-score the 12 s subset by time (per subject, channel, frequency) — the same
    # standardisation the decomposition used, here only for the fixed-mask rows.
    ztracks = {
        condition.value: zscore_by_time(raw_tracks[condition.value])
        for condition in CONDITIONS_TO_POOL
    }
    for condition in CONDITIONS_TO_POOL:
        cond = condition.value
        for i, participant in enumerate(projected_participants):
            # IVA rows: the STORED z-scored source of THIS recording, all fitting onsets.
            iva_src = tf_maps[store_row[(participant, cond)]]  # (K, F, T_full)
            for k in range(n_components):
                band = iva_src[k][test_freq_bins].mean(axis=0)  # (T_full,)
                windows = _cut_windows(band, sidecar_onsets[cond])
                if windows is not None:
                    value_z[cond][i, k] = np.median(_reduce_window(windows))
                    course_z[cond][i, k] = np.median(windows, axis=0)
            # Mask rows: the mask applied to the z-scored 12 s subset.
            x = ztracks[cond][cache_row[cond][participant]]  # (C, F, T_sub)
            for label in MASK_LABELS:
                op = (
                    assr_filter
                    if label == BINARY_FILTER_LABEL
                    else assr_filter_pca_by_condition[cond][i]
                )
                masked = np.tensordot(op, x, axes=([0], [0]))  # (F, T_sub)
                band = masked[test_freq_bins].mean(axis=0)  # (T_sub,)
                col = SOURCE_LABELS.index(label)
                windows = _cut_windows(band, trial_onsets[cond])
                if windows is not None:
                    value_z[cond][i, col] = np.median(_reduce_window(windows))
                    course_z[cond][i, col] = np.median(windows, axis=0)
    value_by_variant["zscored"] = value_z
    course_by_variant["zscored"] = course_z

if "stored_prestim" in TEST_VARIANTS:
    # Nothing left to derive: Step 9b already collapsed frequency, cut the trials and
    # referenced each one to its own baseline. All that remains is the same time-window
    # reduction and median-over-trials the other variants get. The per-source arrays
    # carry DIFFERENT trial counts (the references are the 12 s subset, the ICs the whole
    # recording), which is why Step 9b keeps them per source rather than in one
    # (P, S, N, W) block.
    value_s = {
        condition.value: np.full((n_test_participants, n_sources), np.nan)
        for condition in CONDITIONS_TO_POOL
    }
    course_s = {
        condition.value: np.full((n_test_participants, n_sources, EPOCH_LEN), np.nan)
        for condition in CONDITIONS_TO_POOL
    }
    for condition in CONDITIONS_TO_POOL:
        cond = condition.value
        for label, arr in stored_trials_z[TEST_SELECTION][cond].items():  # (P, N, W)
            col = SOURCE_LABELS.index(label)
            value_s[cond][:, col] = np.nanmedian(_reduce_window(arr), axis=1)
            course_s[cond][:, col] = np.nanmedian(arr, axis=1)
    value_by_variant["stored_prestim"] = value_s
    course_by_variant["stored_prestim"] = course_s

print(
    f"\nBuilt value (P={n_test_participants} x S={n_sources}) and epoch course for "
    f"variant(s): {list(value_by_variant)}"
)


# ── Re-anchor each RECORDING's component polarity to the ASSR area ──
# The topography is per recording here, so the anchoring sign is resolved per
# (participant, condition) from that recording's own pattern. It still reads only the
# fixed spatial reference (the pattern's projection onto the ASSR electrodes), never the
# tested response, so it orients "higher = more ASSR-area power" for every row without
# looking at the quantity being contrasted.
# Per (participant, condition, COMPONENT): this variant estimates one decomposition per
# RECORDING, so a participant's two conditions are separate topographies and may flip
# differently — and every component carries its own independent sign, so one flip shared
# across a participant's ICs would be wrong.
#
# The anchor is corr(topography, 0/1 ASSR mask) across the whole channel axis. Because
# cov(pattern, mask) = f(1-f)(mean_inside - mean_outside) for a binary mask, that asks
# whether the pattern is more positive over the ASSR area THAN OVER THE REST OF THE
# HEAD, so a pattern riding on a global offset cannot flip it — which the older
# mean-over-the-mask anchor could not guarantee. It still reads only the topography and
# never the tested response, so it cannot manufacture a condition contrast.
#
# Every pair is flipped on the sign of that correlation however small: declining to flip
# a weak one keeps the run's arbitrary sign rather than avoiding a choice. The weak ones
# earn a count, carried onto the Step 11 figures.
patterns_by_condition = {
    condition.value: np.stack(
        [
            channel_patterns[store_row[(p, condition.value)]]
            for p in projected_participants
        ]
    )
    for condition in CONDITIONS_TO_POOL
}  # each (P, K, C)

flip_by_condition, strength_by_condition = {}, {}
for condition in CONDITIONS_TO_POOL:
    ic_flip, strength = at.polarity_flip(
        patterns_by_condition[condition.value], assr_mask
    )
    strength_by_condition[condition.value] = strength
    f = np.ones((n_test_participants, n_sources))
    if ALIGN_POLARITY_TO_MASK:
        # The mask rows need no flip: both references are positively aligned with the
        # ASSR area by construction, so "higher = more power there" already holds.
        f[:, :n_components] = ic_flip
    flip_by_condition[condition.value] = f

polarity_weak_by_condition = {
    c.value: at.polarity_weak_count(strength_by_condition[c.value])
    for c in CONDITIONS_TO_POOL
}
POLARITY_NOTE = (
    "sign anchor corr(topography, ASSR mask): "
    + ", ".join(
        f"{c.value} {polarity_weak_by_condition[c.value][0]}/"
        f"{polarity_weak_by_condition[c.value][1]} weak"
        for c in CONDITIONS_TO_POOL
    )
    + f" (|corr| < {at.POLARITY_CORR_FLOOR})"
    if ALIGN_POLARITY_TO_MASK
    else "polarity anchor OFF - component signs are the run's own"
)

value_by_variant = {
    var: {c: flip_by_condition[c] * arr for c, arr in vv.items()}
    for var, vv in value_by_variant.items()
}
course_by_variant = {
    var: {c: flip_by_condition[c][:, :, None] * arr for c, arr in cc.items()}
    for var, cc in course_by_variant.items()
}

print("\nPolarity anchor — corr(topography, ASSR mask), per recording:")
for condition in CONDITIONS_TO_POOL:
    st = strength_by_condition[condition.value]
    fl = flip_by_condition[condition.value]
    weak, total = polarity_weak_by_condition[condition.value]
    print(
        f"  [{condition.value}]  {'IC':<6} {'flipped':>9} {'median |corr|':>15} {'weak':>9}"
    )
    for k in range(n_components):
        weak_k = int((st[:, k] < at.POLARITY_CORR_FLOOR).sum())
        print(
            f"        IC {k + 1:<3} {int((fl[:, k] < 0).sum()):>6}/{n_test_participants} "
            f"{np.median(st[:, k]):>15.3f} {f'{weak_k}/{n_test_participants}':>9}"
        )
    print(
        f"        {weak}/{total} pair(s) decided on |corr| < {at.POLARITY_CORR_FLOOR}; "
        "still flipped, but a wrongly flipped\n        recording CANCELS signal in the "
        "group mean rather than merely widening it."
    )
if not ALIGN_POLARITY_TO_MASK:
    print("  -> ALIGN_POLARITY_TO_MASK is False; 4b's direction is NOT interpretable.")


# ── Exact Wilcoxon signed-rank test ───────────────────────────
def paired_test(differences: np.ndarray, alternative: str) -> dict:
    """Exact Wilcoxon signed-rank test on per-participant differences.

    Paired design: under the null a participant's difference is equally likely to carry
    either sign; Wilcoxon enumerates all 2**P sign assignments over the RANKS of |d|.
    ``method="exact"`` refuses the normal approximation, so the p-value is exact. The
    unit is always the participant (P ~ 12), never the trial, whatever a variant's
    trial count.

    :param differences: One value per participant (non-finite entries are dropped).
    :param alternative: ``"two-sided"``, or ``"greater"`` where the direction was fixed
        in advance and the polarity anchoring makes it meaningful.
    :return: Median difference, robust effect size (median / its own MAD), how many
        participants point positive, the alternative used, and the p-value.
    """
    d = np.asarray(differences, dtype=float)
    d = d[np.isfinite(d)]
    n = d.size
    if n < 2:
        raise ValueError(f"Need at least 2 participants; got {n}.")
    result = wilcoxon(d, alternative=alternative, method="exact")
    spread = 1.4826 * np.median(np.abs(d - np.median(d)))
    return {
        "median": float(np.median(d)),
        "effect": float(np.median(d) / spread) if spread > 0 else np.nan,
        "same sign": f"{int((d > 0).sum())}/{n}",
        "alt": alternative,
        "p": float(result.pvalue),
    }


floor = 2.0 / 2**n_test_participants
print(
    f"\nTest          : exact Wilcoxon signed-rank over {n_test_participants} "
    f"participants (smallest attainable p = {floor:.5f} two-sided, {floor / 2:.5f} "
    "one-sided)"
)

mask_indices = {label: SOURCE_LABELS.index(label) for label in MASK_LABELS}
mask_index = mask_indices[BINARY_FILTER_LABEL]  # the FULL mask; kept for back-compat
ic_labels = [s for s in SOURCE_LABELS if s not in MASK_LABELS]


def _run_tests(value: dict) -> tuple:
    """4b (Placebo-Psilocybin per source) and 4c (IC vs each reference) for one variant."""
    rows = []
    for s, source in enumerate(SOURCE_LABELS):
        d = value[c0][:, s] - value[c1][:, s]
        rows.append({"source": source, **paired_test(d, CONTRAST_ALTERNATIVE)})
    contrast = pd.DataFrame(rows)

    versus = {}
    for mask_label in MASK_LABELS:
        m = mask_indices[mask_label]
        mask_contrast = value[c0][:, m] - value[c1][:, m]
        rr = []
        for source in ic_labels:
            s = SOURCE_LABELS.index(source)
            ic_contrast = value[c0][:, s] - value[c1][:, s]
            rr.append(
                {
                    "source": source,
                    "IC contrast": float(np.nanmedian(ic_contrast)),
                    **paired_test(ic_contrast - mask_contrast, "two-sided"),
                }
            )
        versus[mask_label] = pd.DataFrame(rr)
    return contrast, versus


# ── Run the full family on every variant ──────────────────────
results_by_variant: dict[str, dict] = {}
for variant in TEST_VARIANTS:
    contrast, versus = _run_tests(value_by_variant[variant])
    results_by_variant[variant] = {"contrast": contrast, "versus": versus}

    print(
        f"\n{'=' * 74}\nVARIANT: {variant}   ({TEST_SELECTION}, window {test_window_desc})\n{'=' * 74}"
    )
    print(f"\n4b — {c0} minus {c1}, paired over {n_test_participants} participants:")
    display(contrast.set_index("source").round({"median": 4, "effect": 3, "p": 5}))
    print(
        f"  Uncorrected p over {len(SOURCE_LABELS)} tests — read a row alongside "
        "`effect` and `same sign`, not on its p alone."
    )
    for mask_label in MASK_LABELS:
        vm = versus[mask_label]
        m = mask_indices[mask_label]
        mc = value_by_variant[variant][c0][:, m] - value_by_variant[variant][c1][:, m]
        print(
            f"\n4c vs {mask_label} — (IC contrast) - (reference contrast); "
            f"0 = as good as the reference."
            f"\n     {mask_label} own contrast: median {np.nanmedian(mc):+.4f}, "
            f"{int((mc > 0).sum())}/{n_test_participants} participants positive."
        )
        display(
            vm.set_index("source").round(
                {"IC contrast": 4, "median": 4, "effect": 3, "p": 5}
            )
        )
        better = int((vm["median"] > 0).sum())
        print(
            f"  Uncorrected p over {len(ic_labels)} tests; every IC tested. "
            f"Direction: {better}/{len(ic_labels)} IC(s) separate the conditions more "
            f"than {mask_label}."
        )

# Back-compatible handles pointing at the first requested variant.
_primary = TEST_VARIANTS[0]
value = value_by_variant[_primary]
contrast = results_by_variant[_primary]["contrast"]
versus_by_mask = results_by_variant[_primary]["versus"]
versus_mask = versus_by_mask[BINARY_FILTER_LABEL]

if floor > 0.01:
    print(
        f"\nNOTE: with {n_test_participants} participants no p can fall below "
        f"{floor:.4f}, so a null result is a statement about the design as much as the "
        "effect."
    )


---
## Step 11 — Summary figures

Two figures, answering the two questions the tests were built for.

**Figure 1 — the response over time, per source.** One panel per spatial filter, both
conditions overlaid. The line is the mean across the 12 participants of their own
median-over-trials time course; the band is the spread **across participants**, never
across trials — trials within a participant are correlated (ICC 0.08–0.14), so a band
drawn from them would look tight while saying nothing about how well the effect
generalises to a new person. The stimulus interval is shaded, and each panel carries its
one-sided *p* from 4b so the picture and the test are never read apart.

The IC panels are drawn **polarity-aligned**, the same flip the tests use: without it a
participant whose component loads negatively on the fronto-central electrodes would
cancel one who loads positively, and the group mean would collapse toward zero for
reasons that have nothing to do with the response.

**Figure 2 — the *p*-value summary**, which is the figure to read first.

- **Left: is Psilocybin lower than Placebo?** One row per source, showing the median
  paired difference `Placebo − Psilocybin` with a bootstrap interval. Positive is the
  predicted direction, so the *p* is one-sided `greater`. The `ASSR-mask` row is the
  fixed-electrode reference the components are judged against.
- **Right: does any component beat that reference?** `IC − ASSR-mask` within each
  condition, two-sided because nothing predicts which way this should go.

Intervals are a percentile bootstrap over participants and are shown to convey spread —
the *p*-values come from the exact Wilcoxon test in Step 10, not from the bootstrap, and
the two can disagree slightly at n = 12. Nothing here is corrected for multiplicity, so
read a marker against its neighbours as much as against α.

**Figure 1** now has one panel per source, including both reference rows, and **Figure 2** carries one *versus* panel per reference (IC − `ASSR-mask (full)` and IC − `ASSR-mask (PCA)`) beside the shared Placebo − Psilocybin column.

**One pair of figures per variant.** `draw_variant_figures` is called for each `TEST_VARIANTS` entry, so Figure 1 (epoch response) and Figure 2 (p-value summary) are produced for `zscored` and `prestim` side by side and saved with a `_<variant>` suffix. When `TEST_STIMULUS_INTERVAL` is set, its custom window is shaded in gold on the time-course panels alongside the paradigm's 0-500 ms interval.


In [ ]:
# ── Summary figures, drawn once per signal variant ────────────
# Two figures per variant: (1) the epoch response per source, both conditions overlaid;
# (2) the p-value summary — Placebo-Psilocybin per source, then one IC-vs-reference forest
# per mask. Everything is polarity-aligned exactly as the tests are (the course arrays
# from Step 10 are already flipped).
VARIANT_UNITS = {
    "prestim": "pre-stimulus SD",
    "zscored": "z-scored over time",
    "stored_prestim": "pre-stimulus SD, stored sources",
}
spread_label = (
    "mean +/- SEM across participants"
    if COURSE_SPREAD == "sem"
    else "median with 25-75 band across participants"
)


def _band(values: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Centre line and spread across the PARTICIPANT axis of a ``(P, W)`` array."""
    if COURSE_SPREAD == "sem":
        centre = np.nanmean(values, axis=0)
        n = np.sum(~np.isnan(values), axis=0)
        half = np.nanstd(values, axis=0, ddof=1) / np.sqrt(np.maximum(n, 1))
        return centre, centre - half, centre + half
    if COURSE_SPREAD == "iqr":
        return (
            np.nanmedian(values, axis=0),
            np.nanpercentile(values, 25, axis=0),
            np.nanpercentile(values, 75, axis=0),
        )
    raise ValueError(f"COURSE_SPREAD must be 'sem' or 'iqr'; got {COURSE_SPREAD!r}.")


def _bootstrap_ci(differences: np.ndarray) -> tuple[float, float]:
    """Percentile bootstrap interval for the median of *differences*, over participants."""
    d = np.asarray(differences, dtype=float)
    d = d[np.isfinite(d)]
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    draws = np.median(d[rng.integers(0, d.size, size=(N_BOOTSTRAP, d.size))], axis=1)
    return tuple(np.percentile(draws, [100 * ALPHA / 2, 100 * (1 - ALPHA / 2)]))


def draw_variant_figures(variant: str) -> None:
    """Both summary figures for one signal variant."""
    course = course_by_variant[variant]
    contrast = results_by_variant[variant]["contrast"]
    versus_by_mask = results_by_variant[variant]["versus"]
    value = value_by_variant[variant]
    contrast_by_source = contrast.set_index("source")
    units = VARIANT_UNITS.get(variant, variant)
    c0 = CONDITIONS_TO_POOL[0].value
    c1 = CONDITIONS_TO_POOL[1].value

    # ===== Figure 1 — per-source response over time =====
    n_src = len(SOURCE_LABELS)
    ncols = min(4, n_src)
    nrows = int(np.ceil(n_src / ncols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(5.1 * ncols, 3.7 * nrows),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    axflat = axes.flat
    for s, source in enumerate(SOURCE_LABELS):
        ax = axflat[s]
        ax.axvspan(
            0.0, AssrEpoch.STIMULUS_DURATION_S, color="0.55", alpha=0.11, lw=0, zorder=0
        )
        if TEST_STIMULUS_INTERVAL is not None:
            ax.axvspan(
                *TEST_STIMULUS_INTERVAL, color="#B8860B", alpha=0.10, lw=0, zorder=0
            )
        ax.axhline(0.0, color="0.45", lw=0.8, zorder=1)
        ax.axvline(0.0, color="0.35", lw=0.9, ls="--", zorder=1)
        for condition in CONDITIONS_TO_POOL:
            colour = CONDITION_COLORS[condition.value]
            centre, low, high = _band(course[condition.value][:, s])
            ax.fill_between(
                epoch_times, low, high, color=colour, alpha=0.18, lw=0, zorder=2
            )
            ax.plot(
                epoch_times,
                centre,
                color=colour,
                lw=1.9,
                zorder=3,
                label=condition.value,
            )

        is_reference = source in MASK_LABELS
        row = contrast_by_source.loc[source]
        ax.set_title(
            f"{source}{'  (reference)' if is_reference else ''}",
            fontsize=12,
            fontweight="bold" if is_reference else "normal",
            loc="left",
        )
        ax.text(
            0.985,
            0.955,
            f"{c0} > {c1}\np = {row['p']:.3f}   {row['same sign']}",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=9,
            family="monospace",
            color="0.25" if row["p"] > ALPHA else "black",
            bbox=dict(
                boxstyle="round,pad=0.32",
                facecolor="white",
                edgecolor="0.75" if row["p"] > ALPHA else "black",
                alpha=0.88,
                lw=1.2 if row["p"] <= ALPHA else 0.8,
            ),
        )
        if s + ncols >= n_src:
            ax.set_xlabel("Time from stimulus onset (s)")
        if s % ncols == 0:
            ax.set_ylabel(f"{TEST_SELECTION} power\n({units})")
    for j in range(n_src, nrows * ncols):
        axflat[j].axis("off")

    axflat[0].legend(loc="lower right", frameon=True, fontsize=10)
    fig.suptitle(
        f"{TEST_SELECTION} response per spatial filter — variant: {variant} — "
        f"{spread_label}, {n_test_participants} participants, test window "
        f"{test_window_desc}\n{POLARITY_NOTE}",
        y=1.0,
        fontsize=12.5,
    )
    fig.tight_layout()
    _save(fig, f"trial_course_by_source_{TEST_SELECTION}_{variant}")
    plt.show()

    # ===== Figure 2 — the p-value summary =====
    fig, axes = plt.subplots(
        1,
        1 + len(MASK_LABELS),
        figsize=(7.6 + 5.9 * len(MASK_LABELS), 5.0),
        gridspec_kw={"width_ratios": [1.0] + [1.12] * len(MASK_LABELS)},
    )
    ax_contrast = axes[0]
    versus_axes = axes[1:]

    order = list(SOURCE_LABELS)
    differences = {
        source: value[c0][:, SOURCE_LABELS.index(source)]
        - value[c1][:, SOURCE_LABELS.index(source)]
        for source in order
    }
    intervals = {source: _bootstrap_ci(d) for source, d in differences.items()}

    edges = [bound for pair in intervals.values() for bound in pair]
    edges += [contrast_by_source.loc[source, "median"] for source in order]
    edges += list(np.nanpercentile(np.concatenate(list(differences.values())), [8, 92]))
    span = np.nanmax(edges) - np.nanmin(edges)
    x_lo, x_hi = np.nanmin(edges) - 0.16 * span, np.nanmax(edges) + 0.16 * span

    for i, source in enumerate(order):
        y = len(order) - 1 - i
        d = differences[source]
        low, high = intervals[source]
        row = contrast_by_source.loc[source]
        significant = row["p"] <= ALPHA
        colour = "#1B5E20" if significant else "0.45"

        inside = (d >= x_lo) & (d <= x_hi)
        ax_contrast.scatter(
            d[inside],
            np.full(int(inside.sum()), y),
            s=13,
            color=colour,
            alpha=0.3,
            zorder=2,
            lw=0,
        )
        for value_off in d[(d < x_lo) | (d > x_hi)]:
            ax_contrast.scatter(
                [x_hi if value_off > x_hi else x_lo],
                [y],
                s=26,
                color=colour,
                alpha=0.55,
                marker=">" if value_off > x_hi else "<",
                zorder=2,
                lw=0,
            )
        ax_contrast.plot(
            [low, high], [y, y], color=colour, lw=2.4, zorder=3, solid_capstyle="round"
        )
        ax_contrast.scatter(
            [row["median"]],
            [y],
            s=95,
            color=colour,
            zorder=4,
            marker="D" if source in MASK_LABELS else "o",
            edgecolor="white",
            linewidth=1.1,
        )
        ax_contrast.text(
            1.005,
            y,
            f"p={row['p']:.3f}",
            transform=ax_contrast.get_yaxis_transform(),
            va="center",
            ha="left",
            fontsize=9.5,
            family="monospace",
            fontweight="bold" if significant else "normal",
            color=colour,
        )

    ax_contrast.set_xlim(x_lo, x_hi)
    ax_contrast.axvline(0.0, color="0.3", lw=1.0, zorder=1)
    ax_contrast.set_yticks(range(len(order)))
    ax_contrast.set_yticklabels(list(reversed(order)))
    for tick, source in zip(ax_contrast.get_yticklabels(), reversed(order)):
        if source in MASK_LABELS:
            tick.set_fontweight("bold")
    ax_contrast.set_xlabel(f"{c0} - {c1}  ({units})")
    ax_contrast.set_title(
        "Is Psilocybin lower than Placebo?\none-sided, positive = predicted direction",
        loc="left",
        fontsize=12,
    )

    def _draw_versus(ax, mask_label: str) -> None:
        m = mask_indices[mask_label]
        mask_contrast = value[c0][:, m] - value[c1][:, m]
        ic_order = [s for s in SOURCE_LABELS if s not in MASK_LABELS]
        versus_indexed = versus_by_mask[mask_label].set_index("source")
        interaction = {
            source: (
                value[c0][:, SOURCE_LABELS.index(source)]
                - value[c1][:, SOURCE_LABELS.index(source)]
            )
            - mask_contrast
            for source in ic_order
        }
        inter_ci = {source: _bootstrap_ci(d) for source, d in interaction.items()}
        v_edges = [bound for pair in inter_ci.values() for bound in pair]
        v_edges += [versus_indexed.loc[source, "median"] for source in ic_order]
        v_edges += list(
            np.nanpercentile(np.concatenate(list(interaction.values())), [8, 92])
        )
        v_span = np.nanmax(v_edges) - np.nanmin(v_edges)
        v_lo, v_hi = (
            np.nanmin(v_edges) - 0.16 * v_span,
            np.nanmax(v_edges) + 0.16 * v_span,
        )

        for i, source in enumerate(ic_order):
            y = len(ic_order) - 1 - i
            d = interaction[source]
            low, high = inter_ci[source]
            row = versus_indexed.loc[source]
            significant = row["p"] <= ALPHA
            colour = "#1B5E20" if significant else "0.45"
            inside = (d >= v_lo) & (d <= v_hi)
            ax.scatter(
                d[inside],
                np.full(int(inside.sum()), y),
                s=13,
                color=colour,
                alpha=0.3,
                zorder=2,
                lw=0,
            )
            for value_off in d[(d < v_lo) | (d > v_hi)]:
                ax.scatter(
                    [v_hi if value_off > v_hi else v_lo],
                    [y],
                    s=26,
                    color=colour,
                    alpha=0.55,
                    marker=">" if value_off > v_hi else "<",
                    zorder=2,
                    lw=0,
                )
            ax.plot(
                [low, high],
                [y, y],
                color=colour,
                lw=2.4,
                zorder=3,
                solid_capstyle="round",
            )
            ax.scatter(
                [row["median"]],
                [y],
                s=95,
                color=colour,
                zorder=4,
                edgecolor="white",
                linewidth=1.1,
            )
            ax.text(
                1.005,
                y,
                f"p={row['p']:.3f}",
                transform=ax.get_yaxis_transform(),
                va="center",
                ha="left",
                fontsize=9.5,
                family="monospace",
                fontweight="bold" if significant else "normal",
                color=colour,
            )

        ax.set_xlim(v_lo, v_hi)
        ax.axvline(0.0, color="0.3", lw=1.6, zorder=1)
        ax.set_yticks(range(len(ic_order)))
        ax.set_yticklabels(list(reversed(ic_order)))
        ax.set_xlabel(f"(IC contrast) - ({mask_label} contrast)   ({units})")
        ax.set_title(
            f"Better than {mask_label}?\ntwo-sided; 0 = as good as this reference",
            loc="left",
            fontsize=11.5,
        )

    for ax, mask_label in zip(versus_axes, MASK_LABELS):
        _draw_versus(ax, mask_label)

    n_total_tests = len(contrast) + sum(len(df) for df in versus_by_mask.values())
    fig.suptitle(
        f"Exact Wilcoxon signed-rank, n = {n_test_participants} participants "
        f"(floor p = {floor:.5f}) — variant: {variant} — uncorrected over "
        f"{n_total_tests} tests, {TEST_SELECTION} @ {test_window_desc}",
        y=1.02,
        fontsize=12.5,
    )
    fig.tight_layout()
    _save(fig, f"pvalue_summary_{TEST_SELECTION}_{variant}")
    plt.show()


for variant in TEST_VARIANTS:
    print(f"\n{'#' * 30} FIGURES — variant: {variant} {'#' * 30}")
    draw_variant_figures(variant)

print(
    f"\nBands: {spread_label}. Figure-2 intervals are a {N_BOOTSTRAP:,}-draw percentile "
    "bootstrap over participants (spread only); the p-values are the exact Wilcoxon ones "
    "from Step 10."
)


---
## Step 12 — Is the learned component's SNR lower than the reference's, per condition?

A **magnitude** comparison, distinct from Steps 10-11. Those asked whether a component
separates the *conditions* better than the reference (an interaction). This asks the
plainer question the reference exists to answer: for the same stimulus-window SNR the
tests reduce to, is a learned IC's SNR **significantly lower** than the fixed ASSR
reference's — i.e. does the component recover *less* 40 Hz power over the fronto-central
area than simply averaging the reference electrodes does?

- **Quantity.** Per participant *and condition*, each source's SNR is its stimulus-window
  response from Step 10. In the `prestim` variant this is the baseline-referenced SNR; in
  `zscored` it is the time-standardised response. The values are polarity-anchored to the
  ASSR electrodes, so a lower value is a genuine drop in recovered power, not a sign flip.
- **Test.** Paired exact Wilcoxon, one-sided `less`: `IC SNR − reference SNR < 0`, run
  **separately for Placebo and Psilocybin** (the drug may change how well either filter
  recovers the response). Every (variant, reference, condition, IC) cell is tested —
  `len(TEST_VARIANTS) × len(MASK_LABELS) × 2 × K` tests, uncorrected.
- **Reading it.** One figure, a grid of (variant × reference); within each panel the two
  conditions are drawn as offset rows per IC, coloured by condition (Placebo vs
  Psilocybin), a **filled** marker meaning significant (`p ≤ ALPHA`) and a hollow one not.
  A marker left of zero = that IC's SNR is below the reference in that condition. Weigh
  the `prestim` rows as the cleaner comparison — the IC and mask share the same subset and
  normalisation there — while the `zscored` rows carry the Step-10 caveat that the IC and
  mask are z-scored over different windows.


In [ ]:
# ── Step 12 settings ──────────────────────────────────────────
# Directional hypothesis: a learned component recovers LESS 40 Hz power over the ASSR
# area than the fixed electrode reference, i.e. (IC SNR - reference SNR) < 0. Tested
# SEPARATELY for each condition, since the drug may change how well either filter
# recovers the response.
SNR_TEST_ALTERNATIVE = "less"

# ── The test: each IC's SNR vs each reference's SNR, per variant AND condition ──
# Paired exact Wilcoxon over participants, one-sided. Values are polarity-anchored to the
# ASSR electrodes (Step 10), so "lower" is a genuine drop in recovered power. Every
# (variant, reference, condition, IC) cell is tested; nothing is selected.
n_snr_tests = (
    len(TEST_VARIANTS) * len(MASK_LABELS) * len(CONDITIONS_TO_POOL) * len(ic_labels)
)
snr_forest: dict[tuple, pd.DataFrame] = {}  # (variant, mask_label, condition) -> df
print(
    f"SNR comparison — is a learned IC's SNR {SNR_TEST_ALTERNATIVE} than the reference's?"
    f"\n  {TEST_SELECTION} @ {test_window_desc}, paired over {n_test_participants} "
    f"participants, one-sided, per condition.\n"
)
for variant in TEST_VARIANTS:
    for mask_label in MASK_LABELS:
        m = mask_indices[mask_label]
        combined = []
        for condition in CONDITIONS_TO_POOL:
            snr = value_by_variant[variant][condition.value]  # (P, S), polarity-aligned
            rows = []
            for source in ic_labels:
                s = SOURCE_LABELS.index(source)
                d = snr[:, s] - snr[:, m]  # IC - reference; negative = IC lower
                finite = np.isfinite(d)
                rows.append(
                    {
                        "source": source,
                        "condition": condition.value,
                        "IC SNR": float(np.nanmedian(snr[:, s])),
                        "ref SNR": float(np.nanmedian(snr[:, m])),
                        "IC<ref": f"{int((d[finite] < 0).sum())}/{int(finite.sum())}",
                        **paired_test(d, SNR_TEST_ALTERNATIVE),
                    }
                )
            df = pd.DataFrame(rows)
            snr_forest[(variant, mask_label, condition.value)] = df
            combined.append(df)
        combined_df = pd.concat(combined, ignore_index=True)
        n_sig = int((combined_df["p"] <= ALPHA).sum())
        print(f"[{variant}]  IC SNR vs {mask_label}  (both conditions):")
        display(
            combined_df.set_index(["condition", "source"]).round(
                {"IC SNR": 4, "ref SNR": 4, "median": 4, "effect": 3, "p": 5}
            )
        )
        print(
            f"  {n_sig}/{len(combined_df)} (IC, condition) cell(s) significantly below "
            f"{mask_label} (one-sided {SNR_TEST_ALTERNATIVE}, p <= {ALPHA}). "
            f"Uncorrected over {n_snr_tests} tests.\n"
        )

# ── One figure: IC-minus-reference SNR forest, both conditions overlaid ──
# Grid of (variant x reference); within each panel the two conditions are drawn as
# offset rows per IC, coloured by condition. A filled marker is significant (one-sided
# p <= ALPHA), a hollow one is not.
fig, axes = plt.subplots(
    len(TEST_VARIANTS),
    len(MASK_LABELS),
    figsize=(6.8 * len(MASK_LABELS), 1.5 + 0.82 * len(ic_labels) * len(TEST_VARIANTS)),
    squeeze=False,
)
offsets = np.linspace(0.2, -0.2, len(CONDITIONS_TO_POOL))  # first condition drawn higher
for r, variant in enumerate(TEST_VARIANTS):
    units = VARIANT_UNITS.get(variant, variant)
    for cc, mask_label in enumerate(MASK_LABELS):
        ax = axes[r][cc]
        m = mask_indices[mask_label]
        panel, all_d = {}, []
        for condition in CONDITIONS_TO_POOL:
            snr = value_by_variant[variant][condition.value]
            df = snr_forest[(variant, mask_label, condition.value)].set_index("source")
            diffs = {src: snr[:, SOURCE_LABELS.index(src)] - snr[:, m] for src in ic_labels}
            cis = {src: _bootstrap_ci(dd) for src, dd in diffs.items()}
            panel[condition.value] = (df, diffs, cis)
            all_d.append(np.concatenate(list(diffs.values())))
        edges = []
        for df, diffs, cis in panel.values():
            edges += [b for pair in cis.values() for b in pair]
            edges += [df.loc[src, "median"] for src in ic_labels]
        edges += list(np.nanpercentile(np.concatenate(all_d), [5, 95]))
        span = np.nanmax(edges) - np.nanmin(edges)
        lo, hi = np.nanmin(edges) - 0.16 * span, np.nanmax(edges) + 0.16 * span

        for ci_idx, condition in enumerate(CONDITIONS_TO_POOL):
            colour = CONDITION_COLORS[condition.value]
            df, diffs, cis = panel[condition.value]
            for i, source in enumerate(ic_labels):
                y = len(ic_labels) - 1 - i + offsets[ci_idx]
                low, high = cis[source]
                row = df.loc[source]
                sig = row["p"] <= ALPHA
                ax.plot([low, high], [y, y], color=colour, lw=2.2, zorder=3, solid_capstyle="round", alpha=0.9)
                ax.scatter(
                    [row["median"]], [y], s=72, zorder=4, marker="o",
                    color=colour if sig else "white", edgecolor=colour, linewidth=1.6,
                )
                ax.text(
                    0.985, y, f"p={row['p']:.3f}", transform=ax.get_yaxis_transform(),
                    va="center", ha="right", fontsize=8.5, family="monospace",
                    fontweight="bold" if sig else "normal", color=colour,
                )
        ax.set_xlim(lo, hi)
        ax.axvline(0.0, color="0.3", lw=1.4, zorder=1)
        ax.set_yticks(range(len(ic_labels)))
        ax.set_yticklabels(list(reversed(ic_labels)))
        ax.set_xlabel(f"IC SNR - {mask_label} SNR   ({units})")
        ax.set_title(
            f"{variant} — vs {mask_label}\n"
            f"left of 0 = IC below reference (one-sided {SNR_TEST_ALTERNATIVE})",
            loc="left", fontsize=11,
        )

legend_handles = [
    plt.Line2D([0], [0], marker="o", color=CONDITION_COLORS[c.value], lw=0, markersize=8, label=c.value)
    for c in CONDITIONS_TO_POOL
]
legend_handles += [
    plt.Line2D([0], [0], marker="o", color="0.35", markerfacecolor="0.35", lw=0, markersize=8, label=f"filled: p <= {ALPHA}"),
    plt.Line2D([0], [0], marker="o", color="0.35", markerfacecolor="white", lw=0, markersize=8, label="hollow: n.s."),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=len(legend_handles), fontsize=9, frameon=True, bbox_to_anchor=(0.5, -0.02))
fig.suptitle(
    f"Is the learned component's SNR lower than the ASSR reference's?  "
    f"n = {n_test_participants}, per condition, {TEST_SELECTION} @ {test_window_desc}",
    y=1.02, fontsize=12.5,
)
fig.tight_layout()
_save(fig, f"snr_vs_reference_{TEST_SELECTION}_by_condition")
plt.show()
